In [ ]:
# Restart-safe execution controller for Updated_Shohoj_Pro_Ultra
#
# What this cell does:
# 1. Reuses the same persistent run folder after a kernel restart.
# 2. Records the code cell that was running when the kernel stopped.
# 3. Lets heavy feature/model cells restore completed artifacts or continue from
#    their latest checkpoint when the notebook is run again from the top.
#
# Portable Jupyter notebooks cannot move the UI cursor to another cell after a
# kernel restart. Therefore, rerun from this first cell. Lightweight setup cells
# rebuild the Python namespace, completed heavy work is restored, and the
# interrupted heavy cell continues from its saved progress.

import os
import json
import hashlib
from pathlib import Path
from datetime import datetime

ULTRA_PIPELINE_TOKEN = "Updated_Shohoj_Pro_Ultra_v1"

if Path('/content').exists():
    _default_control_root = Path('/content/.updated_shohoj_pro_ultra_state')
else:
    _default_control_root = Path.cwd() / '.updated_shohoj_pro_ultra_state'

ULTRA_CONTROL_ROOT = Path(
    os.environ.get('TOX21_ULTRA_CONTROL_DIR', str(_default_control_root))
)
ULTRA_CONTROL_ROOT.mkdir(parents=True, exist_ok=True)

ULTRA_ACTIVE_FILE = ULTRA_CONTROL_ROOT / 'active_run.json'
ULTRA_PROGRESS_FILE = ULTRA_CONTROL_ROOT / 'cell_progress.json'


def _ultra_read_json(path, default=None):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except Exception:
        return {} if default is None else default


def _ultra_atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(payload, indent=2, default=str),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def ultra_atomic_save_npz(path, **arrays):
    # Import locally so that the first cell remains independent of the package
    # installation/import cells that follow.
    import numpy as _np

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('wb') as handle:
        _np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def ultra_atomic_torch_save(payload, path):
    import torch as _torch

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    _torch.save(payload, temporary)
    os.replace(temporary, path)


_active = _ultra_read_json(ULTRA_ACTIVE_FILE, {})
_reuse_active = (
    _active.get('pipeline_token') == ULTRA_PIPELINE_TOKEN
    and _active.get('status') != 'completed'
    and bool(_active.get('run_stamp'))
)

if _reuse_active:
    ULTRA_RUN_STAMP = _active['run_stamp']
else:
    ULTRA_RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
    _active = {
        'pipeline_token': ULTRA_PIPELINE_TOKEN,
        'run_stamp': ULTRA_RUN_STAMP,
        'status': 'active',
        'created_at': datetime.now().isoformat(timespec='seconds'),
    }
    _ultra_atomic_write_json(ULTRA_ACTIVE_FILE, _active)

_progress = _ultra_read_json(ULTRA_PROGRESS_FILE, {})
if _progress.get('run_stamp') != ULTRA_RUN_STAMP:
    _progress = {
        'run_stamp': ULTRA_RUN_STAMP,
        'status': 'ready',
        'last_completed_cell': None,
        'running_cell': None,
        'updated_at': datetime.now().isoformat(timespec='seconds'),
    }
    _ultra_atomic_write_json(ULTRA_PROGRESS_FILE, _progress)

ULTRA_RESUME_TARGET = (
    _progress.get('running_cell')
    if _progress.get('status') == 'running'
    else None
)


def ultra_bind_output_dir(output_dir):
    state = _ultra_read_json(ULTRA_ACTIVE_FILE, {})
    state.update({
        'pipeline_token': ULTRA_PIPELINE_TOKEN,
        'run_stamp': ULTRA_RUN_STAMP,
        'status': 'active',
        'output_dir': str(Path(output_dir)),
        'updated_at': datetime.now().isoformat(timespec='seconds'),
    })
    _ultra_atomic_write_json(ULTRA_ACTIVE_FILE, state)


def ultra_mark_run_complete():
    state = _ultra_read_json(ULTRA_ACTIVE_FILE, {})
    state.update({
        'status': 'completed',
        'completed_at': datetime.now().isoformat(timespec='seconds'),
    })
    _ultra_atomic_write_json(ULTRA_ACTIVE_FILE, state)

    progress = _ultra_read_json(ULTRA_PROGRESS_FILE, {})
    progress.update({
        'status': 'completed',
        'running_cell': None,
        'updated_at': datetime.now().isoformat(timespec='seconds'),
    })
    _ultra_atomic_write_json(ULTRA_PROGRESS_FILE, progress)


def _ultra_cell_hash(source):
    normalized = str(source).strip().replace('\r\n', '\n')
    return hashlib.sha1(normalized.encode('utf-8')).hexdigest()


ULTRA_CELL_HASH_MAP = {'41fe1edf74d1d2cc05aeb634e5b9ce1cd9e15282': {'index': 4, 'section': '0. Execution order'}, '087487e8077ae6ee1a8d4e76c94c050d6f77db73': {'index': 6, 'section': '1. Imports and reproducibility settings'}, 'e44b23ab47bfd9143fea34406d213928d18539c3': {'index': 7, 'section': '1. Imports and reproducibility settings'}, '4b521acbe352e097a2c385efb0ca1ff8c608bf70': {'index': 8, 'section': '1. Imports and reproducibility settings'}, 'c83b00af0982a08728c6d5c4fbc79dcbeea58ccf': {'index': 9, 'section': '1. Imports and reproducibility settings'}, '59ddbbd81331d5ff14ef20f287d66c622c0b681b': {'index': 10, 'section': '1. Imports and reproducibility settings'}, '399f2086140909ab4259c5d98c302a525df18700': {'index': 12, 'section': '2. Project folders and dataset location'}, '7041a575550f046a61a8eb5f5133315a789dbac4': {'index': 13, 'section': '2. Project folders and dataset location'}, 'fc136ec59e9724f47040bf5a3d5f5e186c6e82bc': {'index': 15, 'section': '3. Load the Tox21 dataset and validate the schema'}, 'ca76ce6ecd55edc15041783ef808b5f25d1ec1ba': {'index': 16, 'section': '3. Load the Tox21 dataset and validate the schema'}, '6eb827fe110e9a5788fb2ce9ce682dcd3d2e8bec': {'index': 18, 'section': '4. Dataset audit: missing labels and class imbalance'}, '843c5a3950b75e4f2a9347086b4205063fc87663': {'index': 19, 'section': '4. Dataset audit: missing labels and class imbalance'}, 'ce5671b698e50d7fd85c48a55fb81cdba4484b44': {'index': 20, 'section': '4. Dataset audit: missing labels and class imbalance'}, '746711a1191b0382a9a2c7b8c44aa029526d8f3c': {'index': 21, 'section': '4. Dataset audit: missing labels and class imbalance'}, '77d4634865aaacd641abde0b1981c3e83ec305d8': {'index': 23, 'section': '5. Chemical preprocessing'}, 'cc52be2a1a6e6ca3fa476d36e60299033a3a6ac0': {'index': 24, 'section': '5. Chemical preprocessing'}, 'c556d9a15f5b6cbac6cce01c89ee90f982e948fc': {'index': 25, 'section': '5. Chemical preprocessing'}, '9a56de281b53e968eddabb994397c5bc53ec9f3b': {'index': 27, 'section': '6. Molecular feature engineering'}, '247e7cc45e4c316014b0d1b1182c79f5f2ccfd09': {'index': 28, 'section': '6. Molecular feature engineering'}, '6dfb18ed0837cd79081dadbb5245564ad8d3e541': {'index': 29, 'section': '6. Molecular feature engineering'}, '770d9c72eca9c7ebd75a94ba598fab7db2cb12a9': {'index': 30, 'section': '6. Molecular feature engineering'}, 'd38e4e50e97cf0957ab9ab1521b18e49d4d9f9c6': {'index': 31, 'section': '6. Molecular feature engineering'}, '6f173e10c7c263dc808830725b7aa756c1640c6c': {'index': 32, 'section': '6. Molecular feature engineering'}, 'd1b1dfaf434406b717cd7e2260841925a0d4e060': {'index': 34, 'section': '7. Train, validation and test split'}, '7212ccabe48df65e4880b83b76c7fe146e364b64': {'index': 35, 'section': '7. Train, validation and test split'}, 'fb74573f201b658fdd5b913a2ef5ba402b1955b0': {'index': 36, 'section': '7. Train, validation and test split'}, '0116f2af82c77db836f320cfe42a7c8cf2f5a71b': {'index': 38, 'section': '7.1 Bemis-Murcko scaffold-overlap audit'}, '2f51cbab340c6d0a0586b44fd1419259690ba4d8': {'index': 40, 'section': '8. Train-only descriptor filtering, imputation and scaling'}, 'e2d208d9c331ba8935d82ef01011ea0b2f7dfeb3': {'index': 41, 'section': '8. Train-only descriptor filtering, imputation and scaling'}, 'b91c1010c74c84aadd0115c91ebd5f88ee1c53d3': {'index': 43, 'section': '9. Evaluation metrics'}, '4ec442bb069064e7c275d662d68642f625478e6c': {'index': 44, 'section': '9. Evaluation metrics'}, 'cb4bc082438e50054648940a2a90013468b04e27': {'index': 45, 'section': '9. Evaluation metrics'}, 'a75a722678b4bfd91702982cc2f679d6ea1c737f': {'index': 46, 'section': '9. Evaluation metrics'}, '6d10ec09f1b052d816a858b333d80d4d6628c47e': {'index': 47, 'section': '9. Evaluation metrics'}, '6c99c62177ae861652910c0f6808081f0770142a': {'index': 48, 'section': '9. Evaluation metrics'}, '277fd3b462e61fbc26bfe4fcc6caa1c6cc2358c9': {'index': 49, 'section': '9. Evaluation metrics'}, '8697df9dc573ffd8e09d61096af06c74d2182d79': {'index': 51, 'section': '10. Classical machine-learning infrastructure'}, '122b4a11de589947cb656101cb5582d77d78e66f': {'index': 52, 'section': '10. Classical machine-learning infrastructure'}, 'cb094827361e149966781b0acf3ba50dfdca1ce7': {'index': 53, 'section': '10. Classical machine-learning infrastructure'}, 'e1c1816ec3a5749b5ada5647d95ac8460c795dbb': {'index': 54, 'section': '10. Classical machine-learning infrastructure'}, '785e8327995f94e7a8df7b5eb3163f8e880f2510': {'index': 55, 'section': '10. Classical machine-learning infrastructure'}, '85d6dc1776988cfacfa0689deeed0f602397173a': {'index': 58, 'section': 'Model 1 — Regularized Random Forest'}, '50bf7cc81eb27062199a4f88e051e332b870c408': {'index': 59, 'section': 'Model 1 — Regularized Random Forest'}, 'a00a37a1359dd7baf2be05fdbb89cfd74f91a8e5': {'index': 61, 'section': 'Model 2 — Regularized Extra Trees'}, '8f8d05f0e02a03ed496ff01ae353c6b7a0bd8dff': {'index': 62, 'section': 'Model 2 — Regularized Extra Trees'}, '8d3e9723e2f2aa2655ad0bb3f5063f288cc9675e': {'index': 64, 'section': 'Model 3 — Regularized XGBoost'}, '0ab60cb354c120b990fa6a8c00a669de171dd19d': {'index': 65, 'section': 'Model 3 — Regularized XGBoost'}, '472f3636e7489255b2e76df91ce8ae7ddf5cc412': {'index': 67, 'section': 'Optional robustness experiment — Scaffold-group cross-validation'}, '1e8f409d94b1dce46beb15c0f418c4563b39db8e': {'index': 69, 'section': 'Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel'}, 'e6d5f1c3ffd253026fb77301ee193eba90ad059b': {'index': 70, 'section': 'Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel'}, 'a4122f924468f878f146ea40e1bfeb7784fdc67a': {'index': 71, 'section': 'Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel'}, 'acf7dda7a3e6b9c506984da35315491881bc73f9': {'index': 72, 'section': 'Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel'}, 'c06c4dfd0ec48c311ce667838b22f60e5921f42a': {'index': 73, 'section': 'Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel'}, 'c253ef22c174263d988fd7ae893a7c6e5cbb8dda': {'index': 74, 'section': 'Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel'}, 'ac92fc1335c81bc627caad6cf1b213745366320d': {'index': 76, 'section': 'Deep-Learning Infrastructure'}, 'b39a13c4397680c4638f7fed4bfad565e46d114e': {'index': 77, 'section': 'Deep-Learning Infrastructure'}, '4ba5f4a48c9148e483a20450f968e9e215192552': {'index': 78, 'section': 'Deep-Learning Infrastructure'}, '02dd13109de32b2159e26e5e9c40f1ded5b19a59': {'index': 79, 'section': 'Deep-Learning Infrastructure'}, '2245d2805dd138dbd76efeca54b99b8cf3f3c18b': {'index': 80, 'section': 'Deep-Learning Infrastructure'}, '17f26dc90584c4bed278921d55c06731bf87c5a2': {'index': 81, 'section': 'Deep-Learning Infrastructure'}, 'f4caabed5bedd0c6930335f00a6a42fb053d07bd': {'index': 82, 'section': 'Deep-Learning Infrastructure'}, '4870deaad0ab93ca52fb89cf43a0e6aa86caedda': {'index': 83, 'section': 'Deep-Learning Infrastructure'}, '32bff1396a5506a568c6f1d3179eca40f83fcf39': {'index': 85, 'section': 'Model 5 — Strongly Regularized DeepTox-style Multi-task DNN'}, '2e6c6a8202b108985b772d83b0686bbfed52d11a': {'index': 86, 'section': 'Model 5 — Strongly Regularized DeepTox-style Multi-task DNN'}, '54c751b2000ef870adf9ee06b74ca37912bcccaa': {'index': 87, 'section': 'Model 5 — Strongly Regularized DeepTox-style Multi-task DNN'}, '96a95d19c2b4cddd5ecde4da8b88cc67064b0ca8': {'index': 88, 'section': 'Model 5 — Strongly Regularized DeepTox-style Multi-task DNN'}, '6afa816756e31f691cacfa8a9994d8f18c9acc43': {'index': 90, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, 'c930bc2cf66491b49498b26418598c7c5b720d5a': {'index': 91, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, 'f606baa4e652c94886dfb72584811dc72bf25f69': {'index': 92, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, '193ca07440a56bd3b824b40c546576c2c034a377': {'index': 93, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, '2036b2c14d714a56e058095dfb32d85198a556fe': {'index': 94, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, 'c4c571fe2a7a189880d744b6163f0a11d75adf92': {'index': 95, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, 'e3e6bfd04f7c9af95dd2ac146f2acb4e9e0f8fb2': {'index': 96, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, '503c42af5de84f0e32a8f20c935b24272bbd4825': {'index': 97, 'section': 'Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing'}, '9df098963362ca424ab5a0b89d5df2178e5b07a7': {'index': 99, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, '5068099120bc3474409cea0d8062d31ac1fe3035': {'index': 100, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, 'c808eeb4d05145eec2f774ba64e113e36d8699bc': {'index': 101, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, 'd8be6c0f0df9d8eca94fb24ad74f0d35209b2252': {'index': 102, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, 'fbbf3212ca16bae958007418ae0d234cc54db6c8': {'index': 103, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, '764bc958987f0a1d7ff98c0ddf26a2e2716faff0': {'index': 104, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, '94631b5d6b6914fccb2a5c69499b621d9f6c4bc2': {'index': 105, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, '2c4b6cab8bacef44eeeee11fdf6b3a20ee1b9509': {'index': 106, 'section': 'Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid'}, 'ab194579e8feee75df507a08d51ee849663f2ee6': {'index': 108, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, '908472b36b73e859396a7a7a7e136b4cab566986': {'index': 109, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, '38c1e1a62ebf203e1f49463cac17003f861c7db8': {'index': 110, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, 'd5a00f4ffac61228a17d3a71cd4841f71e405e9b': {'index': 111, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, 'de0c3148fe5e8dfd1056e8f9977779a2a8e783c9': {'index': 112, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, '6288a2e3ac23373789f2577bbc60b3ab09516531': {'index': 113, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, 'dfc03a653e0e67ee1b060e3f031d2da10665f8d6': {'index': 114, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, 'a8f62bf8dfb5a8e9c37e85359c3ec816162dc388': {'index': 115, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, 'fc7e3fc41cda52d09282f89a94ab145006e780ed': {'index': 116, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, 'a700ec14ab314527396275e99fe899efbe93f3f4': {'index': 117, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, 'de0f6702cd60dc9a8fbf8a8a7cfd927d24912fdf': {'index': 118, 'section': 'Model 8 — Regularized Multi-channel 2D CNN'}, '98d584a8302e6905e71c2cdf0d2ebbaa2a48fae3': {'index': 120, 'section': 'Model 9 — Strongly Regularized Compact MTL-DNN with Task-Specific Heads'}, '958cc61e3ed672c9a5a0f5cac9bd919d8b721f5a': {'index': 121, 'section': 'Model 9 — Strongly Regularized Compact MTL-DNN with Task-Specific Heads'}, '2b45396514c5903987a7c8d9dc7d9b806c9287c8': {'index': 122, 'section': 'Model 9 — Strongly Regularized Compact MTL-DNN with Task-Specific Heads'}, 'cae6bbaab5f270457c102aeaae5ab8c470633826': {'index': 123, 'section': 'Model 9 — Strongly Regularized Compact MTL-DNN with Task-Specific Heads'}, '69a51506ae67525f545069c7384c2bdbf1076d85': {'index': 124, 'section': 'Model 9 — Strongly Regularized Compact MTL-DNN with Task-Specific Heads'}, '18033a74946087100991e459754d8420a5088051': {'index': 125, 'section': 'Model 9 — Strongly Regularized Compact MTL-DNN with Task-Specific Heads'}, '5179ed8fb351bbf935bf91ff935a94556740a182': {'index': 127, 'section': 'Model 10 — Validation-Selected Correlation-Aware Weighted Rank Ensemble'}, 'ec70c6474288a185fe2736df07c0864943d56104': {'index': 128, 'section': 'Model 10 — Validation-Selected Correlation-Aware Weighted Rank Ensemble'}, 'c3fa51f01d391d00b83e536043393c95d96fc148': {'index': 129, 'section': 'Model 10 — Validation-Selected Correlation-Aware Weighted Rank Ensemble'}, '8256089729ecbe7161fb746cabae8e11de8ff58d': {'index': 130, 'section': 'Model 10 — Validation-Selected Correlation-Aware Weighted Rank Ensemble'}, '2017696fc2ef357ad0cc59f70442296e95643ac1': {'index': 131, 'section': 'Model 10 — Validation-Selected Correlation-Aware Weighted Rank Ensemble'}, 'eee0ea357ca94dfced02163be00149d98c6fc92e': {'index': 132, 'section': 'Model 10 — Validation-Selected Correlation-Aware Weighted Rank Ensemble'}, '06730423f2d054d83a6cb943aeb25e123c0294cb': {'index': 134, 'section': 'Final Results and Visualizations'}, '210bd738282e9d8c5076df5442a9a994ad9cd976': {'index': 135, 'section': 'Final Results and Visualizations'}, '79f2c813a6b63110d8cdbd6afa3719e09d4f1f6c': {'index': 136, 'section': 'Final Results and Visualizations'}, '5e47b1bede3d89478e216900c1f755e7fbc8d821': {'index': 137, 'section': 'Final Results and Visualizations'}, '419d0b6b9847e9125d6c4b410bee32392fbc01e0': {'index': 138, 'section': 'Final Results and Visualizations'}, '91231bc74f5703aa20c7d28b00a0b6b6aacc91f7': {'index': 139, 'section': 'Final Results and Visualizations'}, '4720361c51184b13935d91693cac1d1a9cfdad84': {'index': 140, 'section': 'Final Results and Visualizations'}, '5bc5404c622224354131d64dd83d63e565ea36e6': {'index': 141, 'section': 'Final Results and Visualizations'}, '319c42aa401b6f76034860f8ab0a45fe67b716a4': {'index': 142, 'section': 'Final Results and Visualizations'}, 'a198a35534863a431d703294f707192fd03b8539': {'index': 144, 'section': 'Final export and reproducibility metadata'}, '66e8f0f2f0bab5d5ae29159fcd85276754ef9412': {'index': 145, 'section': 'Final export and reproducibility metadata'}}


def _ultra_pre_run_cell(info):
    raw_cell = getattr(info, 'raw_cell', '')
    cell_meta = ULTRA_CELL_HASH_MAP.get(_ultra_cell_hash(raw_cell))
    if cell_meta is None:
        return

    progress = _ultra_read_json(ULTRA_PROGRESS_FILE, {})
    progress.update({
        'run_stamp': ULTRA_RUN_STAMP,
        'status': 'running',
        'running_cell': cell_meta,
        'updated_at': datetime.now().isoformat(timespec='seconds'),
    })
    _ultra_atomic_write_json(ULTRA_PROGRESS_FILE, progress)


def _ultra_post_run_cell(result):
    raw_cell = getattr(getattr(result, 'info', None), 'raw_cell', '')
    cell_meta = ULTRA_CELL_HASH_MAP.get(_ultra_cell_hash(raw_cell))
    if cell_meta is None:
        return

    if (
        getattr(result, 'error_before_exec', None) is None
        and getattr(result, 'error_in_exec', None) is None
    ):
        progress = _ultra_read_json(ULTRA_PROGRESS_FILE, {})
        progress.update({
            'run_stamp': ULTRA_RUN_STAMP,
            'status': 'ready',
            'last_completed_cell': cell_meta,
            'running_cell': None,
            'updated_at': datetime.now().isoformat(timespec='seconds'),
        })
        _ultra_atomic_write_json(ULTRA_PROGRESS_FILE, progress)


try:
    _ipython = get_ipython()
except NameError:
    _ipython = None

if _ipython is not None:
    # Avoid duplicate hooks when this bootstrap cell is run more than once.
    previous_pre = globals().get('_ULTRA_REGISTERED_PRE_HOOK')
    previous_post = globals().get('_ULTRA_REGISTERED_POST_HOOK')

    if previous_pre is not None:
        try:
            _ipython.events.unregister('pre_run_cell', previous_pre)
        except Exception:
            pass

    if previous_post is not None:
        try:
            _ipython.events.unregister('post_run_cell', previous_post)
        except Exception:
            pass

    _ipython.events.register('pre_run_cell', _ultra_pre_run_cell)
    _ipython.events.register('post_run_cell', _ultra_post_run_cell)
    _ULTRA_REGISTERED_PRE_HOOK = _ultra_pre_run_cell
    _ULTRA_REGISTERED_POST_HOOK = _ultra_post_run_cell

print('=' * 88)
print('Restart-safe controller is ready.')
print('Persistent run stamp:', ULTRA_RUN_STAMP)
if ULTRA_RESUME_TARGET:
    print('Previously interrupted code cell:', ULTRA_RESUME_TARGET)
    print(
        'Run the notebook from this first cell. Completed heavy sections will '
        'be restored, and the interrupted work unit will continue from its '
        'latest checkpoint.'
    )
else:
    print('No interrupted code cell was detected for this run.')
print('=' * 88)


## 0. Restart-safe execution

After a kernel restart, run the notebook again from the first cell. The first cell reuses the same run directory and identifies the interrupted cell. Deterministic features, fold predictions, completed model outputs and epoch checkpoints are reused so that completed heavy work is not repeated.

The notebook UI itself cannot jump automatically to another cell in a portable way. The restart-safe mechanism therefore restores the **work state**, not the visual cursor position.

# Updated_Shohoj_Pro_Ultra — Tox21 Drug Toxicity Prediction

### Beginner-friendly, leakage-aware and strongly regularized ML/DL notebook

This notebook is the restart-safe ultra revision of `Updated_shohoj_Pro.ipynb`. It incorporates the practical code-level recommendations from `Updated_shohoj_Pro.pdf` while keeping the evaluation protocol intentionally simple.

## Evaluation policy

Only the following evaluation measures are reported:

1. **ROC-AUC** — the primary threshold-independent ranking metric.
2. **Accuracy** — calculated with the fixed threshold `0.50`.

PR-AUC, Balanced Accuracy, Sensitivity, Specificity, F1-score, MCC and validation-selected classification thresholds have been removed from the evaluation pipeline.

## Main revisions in this version

- All messages, comments and explanations remain in English.
- A first-cell restart controller and persistent checkpoints now restore completed heavy work and continue interrupted training.
- Global warning suppression was removed; only targeted warning filters are used.
- Iterative multilabel stratification is now required for the main 70/10/20 split.
- An early run-configuration file is saved immediately after the output folder is created.
- Dataset schema checks now verify that every endpoint has both classes in the training and validation splits.
- Chemical preprocessing now exports a stage-by-stage audit table and an original-to-canonical SMILES mapping.
- RDKit descriptor preprocessing now removes train-only near-constant and highly correlated descriptors.
- Sparse-feature retention is exported for reproducibility.
- Random Forest and Extra Trees now expose out-of-bag diagnostics.
- DenseNet PCA now uses `whiten=False` and a variance-retention rule capped at 64 components.
- The CapsNet descriptor vector no longer duplicates LogP as a LogD proxy.
- CapsNet pretraining uses a denoising autoencoder by default because the input contains mixed continuous and binary features.
- DeepTox and Compact MTL-DNN use smaller shared trunks with task-specific heads.
- The boosted hybrid includes chemical-only, visual-only and fused-feature ablation runs.
- The rank ensemble adds correlation-aware candidate filtering to encourage genuine model diversity.
- Prediction matrices are validated and the notebook fails loudly if non-finite predictions appear.
- Bootstrap confidence intervals are exported as optional uncertainty information, while the visible evaluation output remains limited to ROC-AUC and Accuracy.
- Exact package versions used by the run are exported to `requirements_locked.txt`.
- A Bemis-Murcko scaffold-overlap audit is exported. An optional scaffold-group CV section is included but disabled by default because it is computationally expensive.

> Important: this notebook contains revised code, not fabricated results. Run all cells from top to bottom to obtain the new measurements.

## 0. Execution order

Run the notebook from the first cell to the last cell.

After a kernel restart, run the notebook again from the first cell. The same run folder is reused. Lightweight cells rebuild definitions, while deterministic features, completed model outputs and training checkpoints are restored. The interrupted heavy work unit continues from its latest saved progress.

In [ ]:
import os
import sys
import subprocess
import importlib.util

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")

# The notebook installs a package only when the corresponding module is absent.
# Exact versions used by the completed run are exported later to
# requirements_locked.txt for reproducibility.
PACKAGE_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "rdkit": "rdkit",
    "iterstrat": "iterative-stratification",
    "psutil": "psutil",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "torch": "torch",
    "torchvision": "torchvision",
}

missing_packages = [
    pip_name
    for module_name, pip_name in PACKAGE_MAP.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages,
    ])
else:
    print("All required packages are already installed.")

## 1. Imports and reproducibility settings

In [ ]:
import gc
import re
import json
import math
import random
import hashlib
import warnings
import platform
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import psutil
from tqdm.auto import tqdm
from IPython.display import display

In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from scipy.stats import rankdata

from xgboost import XGBClassifier

In [ ]:
try:
    from iterstrat.ml_stratifiers import (
        MultilabelStratifiedShuffleSplit,
        MultilabelStratifiedKFold,
    )
except Exception as exc:
    raise ImportError(
        "iterative-stratification is required for this research notebook. "
        "Install it with: pip install iterative-stratification"
    ) from exc

print("Iterative multilabel stratification is available.")

In [ ]:
from rdkit import Chem, DataStructs, RDLogger, RDConfig
from rdkit.Chem import (
    AllChem,
    MACCSkeys,
    Descriptors,
    Draw,
    Crippen,
    Lipinski,
    rdMolDescriptors,
    ChemicalFeatures,
)
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from torchvision.models import densenet121, DenseNet121_Weights
    TORCHVISION_AVAILABLE = True
except Exception as exc:
    TORCHVISION_AVAILABLE = False
    print("Torchvision is unavailable. The DenseNet hybrid will be skipped.")
    print("Import details:", exc)

In [ ]:
# Do not suppress all warnings globally. Important convergence and API warnings
# should remain visible during a research run.
warnings.filterwarnings(
    "ignore",
    message=".*TypedStorage is deprecated.*",
    category=UserWarning,
)

RDLogger.DisableLog("rdApp.*")

SEED = 42
THRESHOLD = 0.50
CV_FOLDS = 3
TRAIN_RATIO = 0.70
VALID_RATIO = 0.10
TEST_RATIO = 0.20
N_BITS = 2048
RADIUS = 2
TARGET_MEAN_AUC = 0.90

# Bootstrap intervals use only ROC-AUC and Accuracy.
BOOTSTRAP_REPEATS = 300

# The scaffold audit always runs. Full scaffold-group CV is optional because it
# substantially increases runtime.
RUN_SCAFFOLD_GROUP_CV = False

# Hybrid ablation compares chemical-only, visual-only and fused representations.
RUN_HYBRID_ABLATION = True

# Denoising autoencoder is the default CapsNet pretrainer for mixed inputs.
CAPS_PRETRAINER = "autoencoder"

PIPELINE_VERSION = "pro_ultra_v1_auc_accuracy_restart_safe"

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RAM_GB = psutil.virtual_memory().total / (1024 ** 3)
CPU_COUNT = max(1, os.cpu_count() or 1)
N_JOBS = max(1, min(CPU_COUNT - 1, 12))

if RAM_GB < 10:
    RESOURCE_MODE = "low"
elif RAM_GB < 20:
    RESOURCE_MODE = "standard"
else:
    RESOURCE_MODE = "high"

BATCH_TABULAR = 128 if RESOURCE_MODE == "low" else 256
BATCH_IMAGE = (
    12
    if RESOURCE_MODE == "low"
    else (24 if RESOURCE_MODE == "standard" else 48)
)
NUM_WORKERS = (
    0
    if platform.system().lower().startswith("win")
    else min(4, CPU_COUNT // 2)
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("=" * 88)
print("Pipeline version:", PIPELINE_VERSION)
print("Random seed:", SEED)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA GPU was not detected. Deep learning models will run on the CPU.")

print(
    f"RAM: {RAM_GB:.1f} GB | "
    f"CPU threads: {CPU_COUNT} | "
    f"Resource mode: {RESOURCE_MODE}"
)
print("Evaluation metrics: ROC-AUC and Accuracy only")
print("Classification threshold:", THRESHOLD)
print("=" * 88)

## 2. Project folders and dataset location

In [ ]:
# Optional manual path example:
# os.environ["TOX21_DATA_PATH"] = r"D:\Drug_Toxicity\datasets\tox21.csv"

candidate_data_paths = []
if os.environ.get("TOX21_DATA_PATH"):
    candidate_data_paths.append(Path(os.environ["TOX21_DATA_PATH"]))

candidate_data_paths += [
    Path("/content/drive/MyDrive/Drug_Toxicity/datasets/tox21.csv"),
    Path("/content/drive/MyDrive/Drug_Toxicity/datasets/tox21(4).csv"),
    Path("/content/tox21.csv"),
    Path("/content/tox21(4).csv"),
    Path("/mnt/data/tox21(4).csv"),
    Path.cwd() / "datasets" / "tox21.csv",
    Path.cwd() / "datasets" / "tox21(4).csv",
    Path.cwd() / "tox21.csv",
    Path.cwd() / "tox21(4).csv",
]

DATA_PATH = next((path for path in candidate_data_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Tox21 CSV was not found. Set TOX21_DATA_PATH or place the CSV beside the notebook."
    )

In [ ]:
if Path("/content/drive/MyDrive").exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Drug_Toxicity")
elif (Path.cwd() / "datasets").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd() / "Drug_Toxicity_Updated_Shohoj_Pro_Ultra"

PROJECT_TAG = "Updated_Shohoj_Pro_Ultra"
RUN_STAMP = ULTRA_RUN_STAMP
OUTPUT_DIR = PROJECT_ROOT / "outputs" / PROJECT_TAG / RUN_STAMP
MODEL_DIR = OUTPUT_DIR / "models"
FEATURE_DIR = OUTPUT_DIR / "features"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESUME_DIR = OUTPUT_DIR / "resume_state"

for folder in [OUTPUT_DIR, MODEL_DIR, FEATURE_DIR, FIGURE_DIR, RESUME_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ultra_bind_output_dir(OUTPUT_DIR)

RAW_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
RUN_ID = f"{RAW_SHA256[:10]}_seed{SEED}_{PIPELINE_VERSION}"

early_run_configuration = {
    "notebook": "Updated_Shohoj_Pro_Ultra.ipynb",
    "restart_safe_run_stamp": ULTRA_RUN_STAMP,
    "pipeline_version": PIPELINE_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_path": str(DATA_PATH),
    "dataset_sha256": RAW_SHA256,
    "run_id": RUN_ID,
    "seed": SEED,
    "device": str(DEVICE),
    "resource_mode": RESOURCE_MODE,
    "evaluation_metrics": ["ROC-AUC", "Accuracy"],
    "classification_threshold": THRESHOLD,
    "run_scaffold_group_cv": RUN_SCAFFOLD_GROUP_CV,
    "run_hybrid_ablation": RUN_HYBRID_ABLATION,
    "caps_pretrainer": CAPS_PRETRAINER,
}

(OUTPUT_DIR / "run_config_early.json").write_text(
    json.dumps(early_run_configuration, indent=2),
    encoding="utf-8",
)

print("Dataset:", DATA_PATH)
print("Project root:", PROJECT_ROOT)
print("Run output folder:", OUTPUT_DIR)
print("Run ID:", RUN_ID)
print("Early run configuration saved.")
print("Restart-safe resume directory:", RESUME_DIR)


## 3. Load the Tox21 dataset and validate the schema

In [ ]:
EXPECTED_TARGETS = [
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
]

ALIASES = {
    "NR.AhR": "NR-AhR",
    "NR.AR": "NR-AR",
    "NR.AR.LBD": "NR-AR-LBD",
    "NR.Aromatase": "NR-Aromatase",
    "NR.ER": "NR-ER",
    "NR.ER.LBD": "NR-ER-LBD",
    "NR.PPAR.gamma": "NR-PPAR-gamma",
    "SR.ARE": "SR-ARE",
    "SR.ATAD5": "SR-ATAD5",
    "SR.HSE": "SR-HSE",
    "SR.MMP": "SR-MMP",
    "SR.p53": "SR-p53",
}

In [ ]:
df_raw = pd.read_csv(DATA_PATH).rename(columns=ALIASES)

required_columns = set(EXPECTED_TARGETS + ["smiles"])
missing_columns = sorted(required_columns - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

if "mol_id" not in df_raw.columns:
    df_raw["mol_id"] = [f"MOL_{i:06d}" for i in range(len(df_raw))]

for target in EXPECTED_TARGETS:
    df_raw[target] = pd.to_numeric(df_raw[target], errors="coerce")
    invalid_values = set(df_raw[target].dropna().unique()) - {0.0, 1.0}
    if invalid_values:
        raise ValueError(
            f"{target} contains values other than 0, 1 or missing: {invalid_values}"
        )

TARGETS = EXPECTED_TARGETS.copy()

print("Dataset loaded successfully.")
print("Raw dataset shape:", df_raw.shape)
display(df_raw.head(5).style.set_caption("Tox21 Raw Data Preview"))

## 4. Dataset audit: missing labels and class imbalance

In [ ]:
def build_endpoint_summary(frame, targets):
    rows = []
    total_rows = len(frame)

    for target in targets:
        values = frame[target]
        labeled = int(values.notna().sum())
        toxic = int((values == 1).sum())
        non_toxic = int((values == 0).sum())
        missing = total_rows - labeled

        rows.append({
            "Endpoint": target,
            "Labeled": labeled,
            "Toxic (1)": toxic,
            "Non-Toxic (0)": non_toxic,
            "Missing": missing,
            "Missing %": 100.0 * missing / total_rows,
            "Positive Rate %": 100.0 * toxic / labeled if labeled else np.nan,
            "Neg:Pos": non_toxic / max(toxic, 1),
        })

    return pd.DataFrame(rows)

In [ ]:
endpoint_summary_raw = build_endpoint_summary(df_raw, TARGETS)

display(
    endpoint_summary_raw.style
    .format({
        "Missing %": "{:.1f}",
        "Positive Rate %": "{:.1f}",
        "Neg:Pos": "{:.1f}",
    })
    .background_gradient(subset=["Missing %", "Neg:Pos"], cmap="YlOrRd")
    .set_caption("Endpoint-wise Missing Labels and Class Imbalance")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

missing_plot = endpoint_summary_raw.sort_values("Missing %")
axes[0].barh(missing_plot["Endpoint"], missing_plot["Missing %"])
axes[0].set_title("Missing Labels by Endpoint", fontweight="bold")
axes[0].set_xlabel("Missing labels (%)")
for i, value in enumerate(missing_plot["Missing %"]):
    axes[0].text(value + 0.3, i, f"{value:.1f}%", va="center", fontsize=9)

positive_plot = endpoint_summary_raw.sort_values("Positive Rate %")
axes[1].barh(positive_plot["Endpoint"], positive_plot["Positive Rate %"])
axes[1].set_title("Toxic Positive Rate by Endpoint", fontweight="bold")
axes[1].set_xlabel("Positive toxic samples (%)")
for i, value in enumerate(positive_plot["Positive Rate %"]):
    axes[1].text(value + 0.2, i, f"{value:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_dataset_challenges.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(17, 13))

for ax, target in zip(axes.flat, TARGETS):
    counts = (
        df_raw[target]
        .value_counts(dropna=True)
        .reindex([0.0, 1.0], fill_value=0)
    )
    ax.bar(["Non-Toxic", "Toxic"], counts.values)
    ax.set_title(target, fontweight="bold")
    ax.set_ylabel("Count")

    for i, value in enumerate(counts.values):
        ax.text(
            i,
            value + max(counts.values) * 0.015,
            f"{int(value)}",
            ha="center",
            fontsize=8,
        )

plt.suptitle(
    "Per-Task Label Distribution after Missing-Label Masking",
    y=1.02,
    fontweight="bold",
    fontsize=16,
)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_per_task_label_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

## 5. Chemical preprocessing

The preprocessing follows the original notebook logic:

1. Parse each SMILES string.
2. Remove invalid molecules.
3. Keep the largest molecular fragment.
4. Normalize and uncharge the molecule.
5. Sanitize the structure and create canonical SMILES.
6. Remove duplicate canonical molecules.

In [ ]:
largest_fragment = rdMolStandardize.LargestFragmentChooser()
normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()


def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None, None, "invalid_parse"

        mol = largest_fragment.choose(mol)
        mol = normalizer.normalize(mol)
        mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)

        canonical_smiles = Chem.MolToSmiles(mol, canonical=True)
        standardized_mol = Chem.MolFromSmiles(canonical_smiles)
        return canonical_smiles, standardized_mol, "ok"

    except Exception as exc:
        return None, None, str(exc)

In [ ]:
curated_cache_path = OUTPUT_DIR / "curated_tox21.csv"
mapping_cache_path = OUTPUT_DIR / "original_to_canonical_mapping.csv"
audit_cache_path = OUTPUT_DIR / "chemical_preprocessing_audit.csv"
conflict_cache_path = OUTPUT_DIR / "duplicate_label_conflicts.csv"
invalid_cache_path = OUTPUT_DIR / "invalid_smiles.csv"

if curated_cache_path.exists():
    cached_df = pd.read_csv(curated_cache_path)
    required_cached_columns = {
        "mol_id",
        "smiles",
        "canonical_smiles",
        *TARGETS,
    }

    if required_cached_columns.issubset(cached_df.columns):
        cached_df["mol"] = [
            Chem.MolFromSmiles(smiles)
            for smiles in cached_df["canonical_smiles"]
        ]

        if cached_df["mol"].notna().all():
            df = cached_df
            preprocessing_audit = (
                pd.read_csv(audit_cache_path)
                if audit_cache_path.exists()
                else pd.DataFrame()
            )
            duplicate_conflict_table = (
                pd.read_csv(conflict_cache_path)
                if conflict_cache_path.exists()
                else pd.DataFrame()
            )
            print("Restored cleaned chemical dataset from the active run cache.")
        else:
            df = None
    else:
        df = None
else:
    df = None

if df is None:
    df = df_raw.copy()
    standardized_records = []

    for smiles in tqdm(df["smiles"], desc="Standardizing SMILES"):
        standardized_records.append(standardize_smiles(smiles))

    df["canonical_smiles"] = [item[0] for item in standardized_records]
    df["mol"] = [item[1] for item in standardized_records]
    df["standardize_status"] = [item[2] for item in standardized_records]

    rows_before_cleaning = len(df)
    invalid_mask = df["mol"].isna()
    valid_mask_chemistry = ~invalid_mask

    df["is_duplicate_canonical"] = (
        valid_mask_chemistry
        & df.duplicated("canonical_smiles", keep="first")
    )

    df["kept_after_cleaning"] = (
        valid_mask_chemistry
        & ~df["is_duplicate_canonical"]
    )

    invalid_df = df[invalid_mask].copy()
    invalid_df.to_csv(invalid_cache_path, index=False)

    mapping_columns = [
        "mol_id",
        "smiles",
        "canonical_smiles",
        "standardize_status",
        "is_duplicate_canonical",
        "kept_after_cleaning",
    ]

    df[mapping_columns].to_csv(mapping_cache_path, index=False)

    duplicate_conflict_rows = []
    valid_for_conflict_audit = df[valid_mask_chemistry].copy()

    for canonical_smiles, group in valid_for_conflict_audit.groupby(
        "canonical_smiles",
        sort=False,
    ):
        if len(group) < 2:
            continue

        for target in TARGETS:
            observed = sorted(
                group[target].dropna().astype(int).unique().tolist()
            )

            if len(observed) > 1:
                duplicate_conflict_rows.append({
                    "Canonical SMILES": canonical_smiles,
                    "Endpoint": target,
                    "Observed Labels": "|".join(map(str, observed)),
                    "Duplicate Rows": int(len(group)),
                })

    duplicate_conflict_table = pd.DataFrame(duplicate_conflict_rows)
    duplicate_conflict_table.to_csv(conflict_cache_path, index=False)

    preprocessing_audit = pd.DataFrame([
        {
            "Stage": "Raw input",
            "Rows": rows_before_cleaning,
            "Removed at this stage": 0,
        },
        {
            "Stage": "Invalid or unreadable SMILES removed",
            "Rows": int(valid_mask_chemistry.sum()),
            "Removed at this stage": int(invalid_mask.sum()),
        },
        {
            "Stage": "Duplicate canonical SMILES removed",
            "Rows": int(df["kept_after_cleaning"].sum()),
            "Removed at this stage": int(
                df["is_duplicate_canonical"].sum()
            ),
        },
    ])

    preprocessing_audit.to_csv(audit_cache_path, index=False)

    print(
        "Conflicting duplicate endpoint labels:",
        len(duplicate_conflict_table),
    )
    print("Rows before cleaning:", rows_before_cleaning)
    print("Invalid SMILES:", int(invalid_mask.sum()))
    print(
        "Duplicate canonical SMILES:",
        int(df["is_duplicate_canonical"].sum()),
    )

    df = (
        df[df["kept_after_cleaning"]]
        .drop(columns=["is_duplicate_canonical", "kept_after_cleaning"])
        .reset_index(drop=True)
    )

    save_columns = ["mol_id", "smiles", "canonical_smiles", *TARGETS]
    df[save_columns].to_csv(curated_cache_path, index=False)

print("Rows after cleaning:", len(df))

if len(preprocessing_audit):
    display(
        preprocessing_audit.style
        .set_caption("Chemical Preprocessing Audit")
    )


In [ ]:
show_count = min(12, len(df))
show_indices = np.linspace(0, len(df) - 1, show_count, dtype=int)
show_molecules = [df.iloc[i]["mol"] for i in show_indices]
show_legends = [str(df.iloc[i]["mol_id"]) for i in show_indices]

molecule_image = Draw.MolsToGridImage(
    show_molecules,
    molsPerRow=4,
    subImgSize=(280, 220),
    legends=show_legends,
)

display(molecule_image)

## 6. Molecular feature engineering

Classical models and the DeepTox-style network use:

- Morgan ECFP4 fingerprint: 2,048 bits, radius 2
- MACCS structural keys: 167 bits
- All available RDKit 2D descriptors

In [ ]:
def morgan_fp(mol, n_bits=N_BITS, radius=RADIUS):
    values = np.zeros((n_bits,), dtype=np.int8)
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits,
    )
    DataStructs.ConvertToNumpyArray(fingerprint, values)
    return values


def maccs_fp(mol):
    values = np.zeros((167,), dtype=np.int8)
    fingerprint = MACCSkeys.GenMACCSKeys(mol)
    DataStructs.ConvertToNumpyArray(fingerprint, values)
    return values

In [ ]:
descriptor_names = [name for name, _ in Descriptors._descList]
descriptor_functions = [function for _, function in Descriptors._descList]


def rdkit_descriptors(mol):
    values = []

    for function in descriptor_functions:
        try:
            values.append(float(function(mol)))
        except Exception:
            values.append(np.nan)

    return values


def build_feature_matrix_resumable(
    file_name,
    progress_name,
    row_count,
    column_count,
    dtype,
    compute_row,
    description,
    flush_every=64,
):
    data_path = FEATURE_DIR / file_name
    progress_path = RESUME_DIR / progress_name
    expected_shape = (row_count, column_count)

    if data_path.exists():
        try:
            existing = np.load(data_path, mmap_mode="r")
            if existing.shape == expected_shape:
                progress = _ultra_read_json(progress_path, {})
                if progress.get("completed") is True:
                    print(f"Restored {description} from cache.")
                    return existing
        except Exception:
            pass

    if data_path.exists():
        try:
            matrix = np.lib.format.open_memmap(
                data_path,
                mode="r+",
                dtype=dtype,
                shape=expected_shape,
            )
        except Exception:
            data_path.unlink(missing_ok=True)
            matrix = np.lib.format.open_memmap(
                data_path,
                mode="w+",
                dtype=dtype,
                shape=expected_shape,
            )
    else:
        matrix = np.lib.format.open_memmap(
            data_path,
            mode="w+",
            dtype=dtype,
            shape=expected_shape,
        )

    progress = _ultra_read_json(progress_path, {})
    start_row = int(progress.get("completed_rows", 0))
    start_row = max(0, min(start_row, row_count))

    for row_index in tqdm(
        range(start_row, row_count),
        desc=description,
        initial=start_row,
        total=row_count,
    ):
        matrix[row_index] = compute_row(row_index)

        if (
            (row_index + 1) % flush_every == 0
            or row_index + 1 == row_count
        ):
            matrix.flush()
            _ultra_atomic_write_json(
                progress_path,
                {
                    "completed_rows": row_index + 1,
                    "total_rows": row_count,
                    "completed": row_index + 1 == row_count,
                    "updated_at": datetime.now().isoformat(
                        timespec="seconds"
                    ),
                },
            )

    matrix.flush()
    return np.load(data_path, mmap_mode="r")


In [ ]:
X_morgan = build_feature_matrix_resumable(
    file_name="morgan_2048.npy",
    progress_name="morgan_2048_progress.json",
    row_count=len(df),
    column_count=N_BITS,
    dtype="float32",
    compute_row=lambda index: morgan_fp(df.iloc[index]["mol"]),
    description="Morgan ECFP4",
)
print("Morgan shape:", X_morgan.shape)

In [ ]:
X_maccs = build_feature_matrix_resumable(
    file_name="maccs_167.npy",
    progress_name="maccs_167_progress.json",
    row_count=len(df),
    column_count=167,
    dtype="float32",
    compute_row=lambda index: maccs_fp(df.iloc[index]["mol"]),
    description="MACCS",
)
print("MACCS shape:", X_maccs.shape)

In [ ]:
X_desc_array = build_feature_matrix_resumable(
    file_name="rdkit_descriptors.npy",
    progress_name="rdkit_descriptors_progress.json",
    row_count=len(df),
    column_count=len(descriptor_names),
    dtype="float32",
    compute_row=lambda index: rdkit_descriptors(df.iloc[index]["mol"]),
    description="RDKit descriptors",
)

X_desc_raw = pd.DataFrame(
    np.asarray(X_desc_array),
    columns=descriptor_names,
).replace([np.inf, -np.inf], np.nan)

print("Descriptor shape:", X_desc_raw.shape)

In [ ]:
X_fp = np.hstack([X_morgan, X_maccs]).astype(np.float32)
Y = df[TARGETS].to_numpy(dtype=float)

print("Combined binary fingerprint shape:", X_fp.shape)
print("Target matrix shape:", Y.shape)

## 7. Train, validation and test split

The split target is 70% training, 10% validation and 20% testing. Multilabel iterative stratification preserves both positive-label patterns and label-availability patterns across the 12 endpoints.

In [ ]:
all_indices = np.arange(len(df))

Y_positive = np.nan_to_num(Y, nan=0.0).astype(int)
Y_available = np.isfinite(Y).astype(int)
Y_stratification = np.hstack([Y_positive, Y_available]).astype(int)

In [ ]:
split_cache_path = RESUME_DIR / "main_split_indices.npz"

if split_cache_path.exists():
    split_cache = np.load(split_cache_path)
    train_idx = split_cache["train_idx"].astype(int)
    val_idx = split_cache["val_idx"].astype(int)
    test_idx = split_cache["test_idx"].astype(int)
    print("Restored the frozen multilabel split from the active run cache.")
else:
    first_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=TEST_RATIO,
        random_state=SEED,
    )

    train_validation_relative, test_relative = next(
        first_split.split(all_indices, Y_stratification)
    )

    train_validation_indices = all_indices[train_validation_relative]
    test_idx = all_indices[test_relative]

    validation_fraction = VALID_RATIO / (TRAIN_RATIO + VALID_RATIO)

    second_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=validation_fraction,
        random_state=SEED + 1,
    )

    train_relative, validation_relative = next(
        second_split.split(
            train_validation_indices,
            Y_stratification[train_validation_indices],
        )
    )

    train_idx = train_validation_indices[train_relative]
    val_idx = train_validation_indices[validation_relative]

    ultra_atomic_save_npz(
        split_cache_path,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )

Y_train = Y[train_idx]
Y_val = Y[val_idx]
Y_test = Y[test_idx]

print("Multilabel iterative stratification was used.")
print(f"Train:      {len(train_idx):5d} ({100 * len(train_idx) / len(df):.2f}%)")
print(f"Validation: {len(val_idx):5d} ({100 * len(val_idx) / len(df):.2f}%)")
print(f"Test:       {len(test_idx):5d} ({100 * len(test_idx) / len(df):.2f}%)")

In [ ]:
def split_endpoint_counts(indices, split_name):
    rows = []
    split_frame = df.iloc[indices]

    for target in TARGETS:
        rows.append({
            "Split": split_name,
            "Endpoint": target,
            "Non-Toxic": int((split_frame[target] == 0).sum()),
            "Toxic": int((split_frame[target] == 1).sum()),
            "Missing": int(split_frame[target].isna().sum()),
        })

    return pd.DataFrame(rows)


split_distribution = pd.concat([
    split_endpoint_counts(train_idx, "Train"),
    split_endpoint_counts(val_idx, "Validation"),
    split_endpoint_counts(test_idx, "Test"),
], ignore_index=True)

# Fail early if a target lacks both classes in the train or validation split.
for split_name, labels in [
    ("Train", Y_train),
    ("Validation", Y_val),
]:
    for j, target in enumerate(TARGETS):
        available = labels[:, j][np.isfinite(labels[:, j])]
        unique_classes = np.unique(available.astype(int))

        if len(unique_classes) < 2:
            raise ValueError(
                f"{split_name} split does not contain both classes for "
                f"{target}. Found classes: {unique_classes.tolist()}"
            )

print("Every endpoint contains both classes in Train and Validation.")

display(
    split_distribution.head(12)
    .style
    .set_caption("Split Distribution Preview")
)

## 7.1 Bemis-Murcko scaffold-overlap audit

The main split remains the same 70/10/20 multilabel-stratified split so that results stay comparable with the previous notebook.

This audit measures whether the same Bemis-Murcko scaffold appears in multiple splits. High overlap can make random-split performance optimistic for structurally novel compounds.

Acyclic molecules receive molecule-specific scaffold IDs instead of being placed into one giant empty-scaffold group.

In [ ]:
def bemis_murcko_scaffold_id(
    mol,
    canonical_smiles,
):
    scaffold = (
        MurckoScaffold.MurckoScaffoldSmiles(
            mol=mol,
            includeChirality=False,
        )
    )

    if scaffold:
        return scaffold

    digest = hashlib.sha1(
        canonical_smiles.encode("utf-8")
    ).hexdigest()[:16]

    return f"ACYCLIC_{digest}"


SCAFFOLD_IDS = np.asarray([
    bemis_murcko_scaffold_id(
        mol,
        canonical_smiles,
    )
    for mol, canonical_smiles in zip(
        df["mol"],
        df["canonical_smiles"],
    )
], dtype=object)


def scaffold_set(indices):
    return set(
        SCAFFOLD_IDS[indices].tolist()
    )


train_scaffolds = scaffold_set(train_idx)
validation_scaffolds = scaffold_set(val_idx)
test_scaffolds = scaffold_set(test_idx)

scaffold_overlap_audit = pd.DataFrame([
    {
        "Comparison": "Train unique scaffolds",
        "Count": len(train_scaffolds),
    },
    {
        "Comparison": "Validation unique scaffolds",
        "Count": len(validation_scaffolds),
    },
    {
        "Comparison": "Test unique scaffolds",
        "Count": len(test_scaffolds),
    },
    {
        "Comparison": "Train-Validation shared",
        "Count": len(
            train_scaffolds
            & validation_scaffolds
        ),
    },
    {
        "Comparison": "Train-Test shared",
        "Count": len(
            train_scaffolds
            & test_scaffolds
        ),
    },
    {
        "Comparison": "Validation-Test shared",
        "Count": len(
            validation_scaffolds
            & test_scaffolds
        ),
    },
])

scaffold_overlap_audit.to_csv(
    OUTPUT_DIR / "scaffold_overlap_audit.csv",
    index=False,
)

pd.DataFrame({
    "mol_id": df["mol_id"],
    "canonical_smiles": df["canonical_smiles"],
    "bemis_murcko_scaffold": SCAFFOLD_IDS,
}).to_csv(
    OUTPUT_DIR / "compound_scaffold_mapping.csv",
    index=False,
)

display(
    scaffold_overlap_audit.style
    .set_caption(
        "Bemis-Murcko Scaffold-Overlap Audit"
    )
)

## 8. Train-only descriptor filtering, imputation and scaling

All continuous-descriptor preprocessing is learned from the training data only.

The pipeline now performs:

1. Median imputation.
2. Near-constant descriptor removal.
3. Highly correlated descriptor removal.
4. Standard scaling.

The same fitted transformation is then applied to validation and test data. Fold-specific copies are rebuilt inside cross-validation to prevent leakage. Target labels are never imputed.

In [ ]:
def fit_descriptor_preprocessor(
    fit_indices,
    variance_threshold=1e-12,
    correlation_threshold=0.98,
):
    imputer = SimpleImputer(strategy="median")

    descriptor_fit = imputer.fit_transform(
        X_desc_raw.iloc[fit_indices]
    )

    descriptor_variance = np.var(
        descriptor_fit,
        axis=0,
    )

    variance_keep = descriptor_variance > variance_threshold
    descriptor_after_variance = descriptor_fit[:, variance_keep]

    kept_after_variance_indices = np.where(variance_keep)[0]

    if descriptor_after_variance.shape[1] > 1:
        correlation_matrix = np.corrcoef(
            descriptor_after_variance,
            rowvar=False,
        )

        correlation_matrix = np.nan_to_num(
            correlation_matrix,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        upper_triangle = np.triu(
            np.abs(correlation_matrix),
            k=1,
        )

        correlation_drop = (
            upper_triangle > correlation_threshold
        ).any(axis=0)

        correlation_keep = ~correlation_drop
    else:
        correlation_keep = np.ones(
            descriptor_after_variance.shape[1],
            dtype=bool,
        )

    final_descriptor_indices = (
        kept_after_variance_indices[correlation_keep]
    )

    scaler = StandardScaler()
    selected_fit = descriptor_fit[:, final_descriptor_indices]
    selected_fit = scaler.fit_transform(selected_fit)

    preprocessor = {
        "imputer": imputer,
        "scaler": scaler,
        "selected_indices": final_descriptor_indices,
        "selected_names": [
            descriptor_names[index]
            for index in final_descriptor_indices
        ],
        "variance_threshold": variance_threshold,
        "correlation_threshold": correlation_threshold,
        "input_descriptor_count": len(descriptor_names),
        "retained_descriptor_count": len(final_descriptor_indices),
    }

    return selected_fit.astype(np.float32), preprocessor


def transform_descriptors(indices, preprocessor):
    descriptor_values = preprocessor["imputer"].transform(
        X_desc_raw.iloc[indices]
    )

    descriptor_values = descriptor_values[
        :,
        preprocessor["selected_indices"],
    ]

    descriptor_values = preprocessor["scaler"].transform(
        descriptor_values
    )

    return descriptor_values.astype(np.float32)


def build_chemical_features(fit_indices, transform_indices):
    fit_descriptors, preprocessor = fit_descriptor_preprocessor(
        fit_indices
    )

    transformed_descriptors = transform_descriptors(
        transform_indices,
        preprocessor,
    )

    X_fit = np.hstack([
        X_fp[fit_indices],
        fit_descriptors,
    ]).astype(np.float32)

    X_transform = np.hstack([
        X_fp[transform_indices],
        transformed_descriptors,
    ]).astype(np.float32)

    return X_fit, X_transform, preprocessor

In [ ]:
(
    X_train_chem,
    X_val_chem,
    descriptor_preprocessor,
) = build_chemical_features(
    train_idx,
    val_idx,
)

test_descriptors = transform_descriptors(
    test_idx,
    descriptor_preprocessor,
)

X_test_chem = np.hstack([
    X_fp[test_idx],
    test_descriptors,
]).astype(np.float32)

joblib.dump(
    descriptor_preprocessor,
    MODEL_DIR / "chemical_preprocessing.joblib",
)

descriptor_audit = pd.DataFrame({
    "Descriptor": descriptor_names,
    "Retained": [
        index in set(
            descriptor_preprocessor["selected_indices"].tolist()
        )
        for index in range(len(descriptor_names))
    ],
})

descriptor_audit.to_csv(
    OUTPUT_DIR / "descriptor_filter_audit.csv",
    index=False,
)

print(
    "RDKit descriptors retained:",
    descriptor_preprocessor["retained_descriptor_count"],
    "/",
    descriptor_preprocessor["input_descriptor_count"],
)

print("Train feature shape:", X_train_chem.shape)
print("Validation feature shape:", X_val_chem.shape)
print("Test feature shape:", X_test_chem.shape)

## 9. Evaluation metrics

This notebook reports only two evaluation measures:

1. **ROC-AUC** — the primary threshold-independent ranking metric.
2. **Accuracy** — calculated with the fixed classification threshold `0.50`.

Missing labels are ignored. No target label is imputed.

PR-AUC, Balanced Accuracy, Sensitivity, Specificity, F1-score, MCC, confusion-matrix reporting and validation-selected thresholds are intentionally excluded from this version.

Bootstrap 95% confidence intervals are calculated for Mean ROC-AUC and Mean Accuracy using resampling of test compounds.

In [ ]:
def slugify(text):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        text.lower(),
    ).strip("_")


def valid_mask(y_true, y_score=None):
    y_true = np.asarray(y_true, dtype=float)
    mask = np.isfinite(y_true)

    if y_score is not None:
        mask &= np.isfinite(
            np.asarray(y_score, dtype=float)
        )

    return mask


def safe_auc(y_true, y_score):
    mask = valid_mask(y_true, y_score)
    y = np.asarray(y_true, dtype=float)[mask]
    score = np.asarray(y_score, dtype=float)[mask]

    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan

    return float(roc_auc_score(y, score))


def safe_accuracy(
    y_true,
    y_score,
    threshold=THRESHOLD,
):
    mask = valid_mask(y_true, y_score)
    y = np.asarray(y_true, dtype=float)[mask].astype(int)
    score = np.asarray(y_score, dtype=float)[mask]

    if len(y) == 0:
        return np.nan

    prediction = (score >= threshold).astype(int)
    return float(accuracy_score(y, prediction))


def mean_metric(values):
    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    ).to_numpy(dtype=float)

    return (
        float(np.nanmean(values))
        if np.isfinite(values).any()
        else np.nan
    )


def prior_probability(y):
    y = np.asarray(y, dtype=float)

    return (
        float(np.nanmean(y))
        if np.isfinite(y).any()
        else 0.0
    )

In [ ]:
def validate_prediction_matrix(
    matrix,
    expected_shape,
    matrix_name,
):
    matrix = np.asarray(matrix, dtype=float)

    if matrix.shape != expected_shape:
        raise ValueError(
            f"{matrix_name} has shape {matrix.shape}; "
            f"expected {expected_shape}."
        )

    if not np.isfinite(matrix).all():
        invalid_count = int(
            matrix.size - np.isfinite(matrix).sum()
        )

        raise ValueError(
            f"{matrix_name} contains {invalid_count} "
            "non-finite predictions. The run is stopped so that "
            "a failed model cannot silently bias the metrics."
        )

    return np.clip(matrix, 0.0, 1.0)

In [ ]:
def bootstrap_mean_auc_accuracy(
    Y_test_local,
    P_test_local,
    repeats=BOOTSTRAP_REPEATS,
    seed=SEED,
):
    rng = np.random.default_rng(seed)
    number_of_compounds = len(Y_test_local)

    auc_samples = []
    accuracy_samples = []

    for _ in range(repeats):
        sample_indices = rng.integers(
            0,
            number_of_compounds,
            size=number_of_compounds,
        )

        auc_value = mean_metric([
            safe_auc(
                Y_test_local[sample_indices, j],
                P_test_local[sample_indices, j],
            )
            for j in range(len(TARGETS))
        ])

        accuracy_value = mean_metric([
            safe_accuracy(
                Y_test_local[sample_indices, j],
                P_test_local[sample_indices, j],
                THRESHOLD,
            )
            for j in range(len(TARGETS))
        ])

        if np.isfinite(auc_value):
            auc_samples.append(auc_value)

        if np.isfinite(accuracy_value):
            accuracy_samples.append(accuracy_value)

    def interval(values):
        if len(values) < 20:
            return np.nan, np.nan

        lower, upper = np.percentile(
            values,
            [2.5, 97.5],
        )

        return float(lower), float(upper)

    auc_low, auc_high = interval(auc_samples)
    accuracy_low, accuracy_high = interval(
        accuracy_samples
    )

    return {
        "Test Mean AUC 95% CI Low": auc_low,
        "Test Mean AUC 95% CI High": auc_high,
        "Test Mean Accuracy 95% CI Low": accuracy_low,
        "Test Mean Accuracy 95% CI High": accuracy_high,
    }

In [ ]:
def predict_positive_probability(model, X):
    if hasattr(model, "predict_proba"):
        probabilities = np.asarray(model.predict_proba(X))

        if probabilities.ndim == 1:
            return probabilities.astype(float)
        if probabilities.shape[1] == 1:
            return np.zeros(len(X), dtype=float)

        return probabilities[:, 1].astype(float)

    if hasattr(model, "decision_function"):
        score = np.asarray(model.decision_function(X), dtype=float)
        score = np.clip(score, -50, 50)
        return 1.0 / (1.0 + np.exp(-score))

    return np.asarray(model.predict(X), dtype=float)

In [ ]:
def endpoint_metric_table(
    P_train_diagnostic,
    P_val,
    P_test,
):
    rows = []

    for j, target in enumerate(TARGETS):
        rows.append({
            "Endpoint": target,
            "Diagnostic Train AUC": safe_auc(
                Y_train[:, j],
                P_train_diagnostic[:, j],
            ),
            "Validation AUC": safe_auc(
                Y_val[:, j],
                P_val[:, j],
            ),
            "Test AUC": safe_auc(
                Y_test[:, j],
                P_test[:, j],
            ),
            "Test Accuracy": safe_accuracy(
                Y_test[:, j],
                P_test[:, j],
                THRESHOLD,
            ),
            "Test Labeled": int(
                np.isfinite(Y_test[:, j]).sum()
            ),
            "Test Toxic": int(
                np.nansum(Y_test[:, j] == 1)
            ),
            "Test Non-Toxic": int(
                np.nansum(Y_test[:, j] == 0)
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
def history_overfit_flag(history):
    if not history:
        return False

    train_loss = np.asarray(history.get("train_loss", []), dtype=float)
    validation_loss = np.asarray(history.get("val_loss", []), dtype=float)

    if len(train_loss) < 8 or len(validation_loss) < 8:
        return False

    best_validation_epoch = int(np.nanargmin(validation_loss))
    tail_size = min(5, len(validation_loss))

    validation_rebound = (
        np.nanmean(validation_loss[-tail_size:])
        > np.nanmin(validation_loss) * 1.08
    )

    comparison_start = max(0, best_validation_epoch - tail_size + 1)
    train_continues_down = (
        np.nanmean(train_loss[-tail_size:])
        < np.nanmean(
            train_loss[comparison_start:best_validation_epoch + 1]
        ) * 0.97
    )

    return bool(validation_rebound and train_continues_down)


def fitting_status(train_auc, validation_auc, history=None):
    if not np.isfinite(train_auc) or not np.isfinite(validation_auc):
        return "Insufficient data"

    gap = train_auc - validation_auc

    if train_auc < 0.70 and validation_auc < 0.70:
        return "Underfitting"
    if validation_auc - train_auc > 0.07:
        return "Split-unstable"
    if gap > 0.10 or (gap > 0.075 and history_overfit_flag(history)):
        return "Overfitting"
    if gap > 0.06:
        return "Mild overfitting"

    return "Well-fitted"

In [ ]:
MODEL_RESULTS = {}
MODEL_ENDPOINT_RESULTS = {}
MODEL_PREDICTIONS = {}
MODEL_HISTORIES = {}
CV_TABLES = {}
SPARSE_FEATURE_AUDIT = []
FOLD_FEATURE_AUDIT = []


def completed_model_result_paths(model_name):
    model_output_dir = OUTPUT_DIR / slugify(model_name)
    return {
        "directory": model_output_dir,
        "predictions": model_output_dir / "predictions.npz",
        "summary": model_output_dir / "overall_summary.json",
        "endpoint": model_output_dir / "per_endpoint_metrics.csv",
        "history": model_output_dir / "history.json",
    }


def restore_completed_model_result(model_name):
    paths = completed_model_result_paths(model_name)
    required = [
        paths["predictions"],
        paths["summary"],
        paths["endpoint"],
    ]

    if not all(path.exists() for path in required):
        return None

    try:
        predictions = np.load(paths["predictions"])
        previous_summary = json.loads(
            paths["summary"].read_text(encoding="utf-8")
        )
        history = (
            json.loads(paths["history"].read_text(encoding="utf-8"))
            if paths["history"].exists()
            else None
        )

        print(f"Restoring completed model output: {model_name}")

        return register_model_result(
            model_name,
            predictions["diagnostic_train"],
            predictions["validation"],
            predictions["test"],
            history=history,
            notes=previous_summary.get("Notes", ""),
            diagnostic_method=previous_summary.get(
                "Diagnostic Method",
                "Restored completed model output",
            ),
        )
    except Exception as exc:
        print(
            f"Completed output for {model_name} could not be restored; "
            f"the model will be recomputed. Details: {exc}"
        )
        return None


def register_model_result(
    model_name,
    P_train_diagnostic,
    P_val,
    P_test,
    history=None,
    notes="",
    diagnostic_method="Diagnostic train prediction",
):
    P_train_diagnostic = validate_prediction_matrix(
        P_train_diagnostic,
        (len(train_idx), len(TARGETS)),
        f"{model_name} diagnostic-train predictions",
    )

    P_val = validate_prediction_matrix(
        P_val,
        (len(val_idx), len(TARGETS)),
        f"{model_name} validation predictions",
    )

    P_test = validate_prediction_matrix(
        P_test,
        (len(test_idx), len(TARGETS)),
        f"{model_name} test predictions",
    )

    per_endpoint = endpoint_metric_table(
        P_train_diagnostic,
        P_val,
        P_test,
    )

    train_auc = mean_metric(
        per_endpoint["Diagnostic Train AUC"]
    )

    validation_auc = mean_metric(
        per_endpoint["Validation AUC"]
    )

    test_auc = mean_metric(
        per_endpoint["Test AUC"]
    )

    test_accuracy = mean_metric(
        per_endpoint["Test Accuracy"]
    )

    confidence_intervals = bootstrap_mean_auc_accuracy(
        Y_test,
        P_test,
        repeats=BOOTSTRAP_REPEATS,
        seed=SEED + sum(ord(char) for char in model_name),
    )

    status = fitting_status(
        train_auc,
        validation_auc,
        history=history,
    )

    summary = {
        "Model": model_name,
        "Diagnostic Train Mean AUC": train_auc,
        "Validation Mean AUC": validation_auc,
        "Test Mean AUC": test_auc,
        "Test Mean Accuracy": test_accuracy,
        **confidence_intervals,
        "Train-Validation AUC Gap": train_auc - validation_auc,
        "Fitting Status": status,
        "Diagnostic Method": diagnostic_method,
        "Target AUC Reached": bool(
            np.isfinite(test_auc)
            and test_auc >= TARGET_MEAN_AUC
        ),
        "Notes": notes,
    }

    model_output_dir = (
        OUTPUT_DIR / slugify(model_name)
    )

    model_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        model_output_dir / "predictions.npz",
        diagnostic_train=P_train_diagnostic,
        validation=P_val,
        test=P_test,
    )

    per_endpoint.to_csv(
        model_output_dir / "per_endpoint_metrics.csv",
        index=False,
    )

    (
        model_output_dir / "overall_summary.json"
    ).write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )

    if history is not None:
        (
            model_output_dir / "history.json"
        ).write_text(
            json.dumps(history, indent=2),
            encoding="utf-8",
        )

        MODEL_HISTORIES[model_name] = history

    MODEL_RESULTS[model_name] = summary
    MODEL_ENDPOINT_RESULTS[model_name] = per_endpoint

    MODEL_PREDICTIONS[model_name] = {
        "diagnostic_train": P_train_diagnostic,
        "validation": P_val,
        "test": P_test,
    }

    print("\n" + "=" * 96)
    print(model_name)
    print(
        f"Diagnostic Train Mean AUC : "
        f"{train_auc:.4f}"
    )
    print(
        f"Validation Mean AUC       : "
        f"{validation_auc:.4f}"
    )
    print(
        f"Test Mean AUC             : "
        f"{test_auc:.4f}"
    )
    print(
        f"Test Mean Accuracy        : "
        f"{test_accuracy:.4f}"
    )
    print(f"Fitting Status            : {status}")
    print(f"Diagnostic Method         : {diagnostic_method}")
    print("=" * 96)

    display(
        per_endpoint[[
            "Endpoint",
            "Diagnostic Train AUC",
            "Validation AUC",
            "Test AUC",
            "Test Accuracy",
        ]]
        .style
        .format({
            "Diagnostic Train AUC": "{:.4f}",
            "Validation AUC": "{:.4f}",
            "Test AUC": "{:.4f}",
            "Test Accuracy": "{:.4f}",
        })
        .background_gradient(
            subset=["Test AUC"],
            cmap="RdYlGn",
            vmin=0.5,
            vmax=1.0,
        )
        .set_caption(
            f"{model_name} — ROC-AUC and Accuracy"
        )
    )

    return summary, per_endpoint

## 10. Classical machine-learning infrastructure

All classical models use 3-fold out-of-fold predictions for the diagnostic training AUC. Final endpoint models are then fitted on the full 70% training split. Validation data are used only for early stopping or model selection, and test labels are not used during training.

In [ ]:
def build_chemical_features_for_fold(
    fit_indices,
    transform_indices,
):
    X_fit, X_transform, preprocessor = (
        build_chemical_features(
            fit_indices,
            transform_indices,
        )
    )

    fold_hash = hashlib.sha1(
        np.asarray(
            fit_indices,
            dtype=np.int64,
        ).tobytes()
    ).hexdigest()[:12]

    FOLD_FEATURE_AUDIT.append({
        "Fit Index Hash": fold_hash,
        "Fit Rows": int(len(fit_indices)),
        "Transform Rows": int(
            len(transform_indices)
        ),
        "Input Descriptors": int(
            preprocessor[
                "input_descriptor_count"
            ]
        ),
        "Retained Descriptors": int(
            preprocessor[
                "retained_descriptor_count"
            ]
        ),
    })

    return X_fit, X_transform

In [ ]:
def fit_endpoint_estimator(model, X_train, y_train, X_validation=None, y_validation=None):
    if (
        isinstance(model, XGBClassifier)
        and X_validation is not None
        and y_validation is not None
    ):
        validation_mask = np.isfinite(y_validation)

        if (
            validation_mask.sum() >= 10
            and len(np.unique(y_validation[validation_mask])) == 2
        ):
            model.fit(
                X_train,
                y_train,
                eval_set=[(
                    X_validation[validation_mask],
                    y_validation[validation_mask].astype(int),
                )],
                verbose=False,
            )
            return model

    model.fit(X_train, y_train)
    return model

In [ ]:
def multilabel_cv_splits():
    fold_targets = np.hstack([
        np.nan_to_num(
            Y_train,
            nan=0.0,
        ).astype(int),
        np.isfinite(Y_train).astype(int),
    ])

    splitter = MultilabelStratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    return list(
        splitter.split(
            np.zeros(
                (len(train_idx), 1),
                dtype=np.float32,
            ),
            fold_targets,
        )
    )


CV_SPLITS = multilabel_cv_splits()

print(
    "Prepared",
    len(CV_SPLITS),
    "multilabel cross-validation folds.",
)

In [ ]:
def run_endpoint_cv(
    model_name,
    model_factory,
    fold_feature_builder,
):
    cache_dir = RESUME_DIR / slugify(model_name)
    cache_dir.mkdir(parents=True, exist_ok=True)

    final_oof_path = cache_dir / "three_fold_oof_complete.npz"
    final_cv_path = cache_dir / "three_fold_cv_complete.csv"

    if final_oof_path.exists() and final_cv_path.exists():
        cached = np.load(final_oof_path)
        P_oof = cached["P_oof"]
        cv_table = pd.read_csv(final_cv_path)

        if P_oof.shape == (len(train_idx), len(TARGETS)):
            CV_TABLES[model_name] = cv_table
            print(f"Restored completed 3-fold OOF predictions: {model_name}")
            return cv_table, P_oof

    progress_path = cache_dir / "three_fold_cv_progress.npz"
    rows_path = cache_dir / "three_fold_cv_rows.json"

    P_oof = np.full(
        (len(train_idx), len(TARGETS)),
        np.nan,
        dtype=float,
    )
    completed_folds = np.zeros(len(CV_SPLITS), dtype=bool)
    fold_rows = []

    if progress_path.exists():
        try:
            progress = np.load(progress_path)
            cached_oof = progress["P_oof"]
            cached_completed = progress["completed_folds"].astype(bool)

            if cached_oof.shape == P_oof.shape:
                P_oof = cached_oof
                completed_folds = cached_completed
                fold_rows = _ultra_read_json(rows_path, [])
                print(f"Resuming 3-fold OOF work: {model_name}")
        except Exception:
            pass

    for fold_index, (train_relative, validation_relative) in enumerate(
        CV_SPLITS
    ):
        if completed_folds[fold_index]:
            continue

        fold_number = fold_index + 1
        train_absolute = train_idx[train_relative]
        validation_absolute = train_idx[validation_relative]

        X_fold_train, X_fold_validation = fold_feature_builder(
            train_absolute,
            validation_absolute,
        )

        Y_fold_train = Y[train_absolute]
        Y_fold_validation = Y[validation_absolute]

        P_fold_validation = np.full(
            (len(validation_relative), len(TARGETS)),
            np.nan,
            dtype=float,
        )

        for j in range(len(TARGETS)):
            y_train_endpoint = Y_fold_train[:, j]
            train_mask = np.isfinite(y_train_endpoint)
            fallback = prior_probability(y_train_endpoint)

            if (
                train_mask.sum() < 10
                or len(np.unique(y_train_endpoint[train_mask])) < 2
            ):
                P_fold_validation[:, j] = fallback
                continue

            model = model_factory(y_train_endpoint[train_mask])
            model = fit_endpoint_estimator(
                model,
                X_fold_train[train_mask],
                y_train_endpoint[train_mask].astype(int),
                X_fold_validation,
                Y_fold_validation[:, j],
            )

            P_fold_validation[:, j] = np.clip(
                predict_positive_probability(model, X_fold_validation),
                0.0,
                1.0,
            )

        P_oof[validation_relative] = P_fold_validation

        fold_auc = mean_metric([
            safe_auc(
                Y_fold_validation[:, j],
                P_fold_validation[:, j],
            )
            for j in range(len(TARGETS))
        ])

        fold_accuracy = mean_metric([
            safe_accuracy(
                Y_fold_validation[:, j],
                P_fold_validation[:, j],
                THRESHOLD,
            )
            for j in range(len(TARGETS))
        ])

        fold_rows = [
            row for row in fold_rows
            if int(row.get("Fold", -1)) != fold_number
        ]
        fold_rows.append({
            "Model": model_name,
            "Fold": fold_number,
            "CV Mean AUC": fold_auc,
            "CV Mean Accuracy": fold_accuracy,
        })
        fold_rows = sorted(fold_rows, key=lambda row: int(row["Fold"]))

        completed_folds[fold_index] = True
        ultra_atomic_save_npz(
            progress_path,
            P_oof=P_oof,
            completed_folds=completed_folds,
        )
        _ultra_atomic_write_json(rows_path, fold_rows)

        print(
            f"{model_name} | Fold {fold_number}: "
            f"AUC={fold_auc:.4f}, Accuracy={fold_accuracy:.4f}"
        )

    cv_table = pd.DataFrame(fold_rows)
    CV_TABLES[model_name] = cv_table

    cv_table.to_csv(
        OUTPUT_DIR / f"{slugify(model_name)}_three_fold_cv.csv",
        index=False,
    )
    cv_table.to_csv(final_cv_path, index=False)
    ultra_atomic_save_npz(final_oof_path, P_oof=P_oof)

    return cv_table, P_oof

In [ ]:
def run_classical_endpoint_model(
    model_name,
    model_factory,
    X_train,
    X_validation,
    X_test,
    fold_feature_builder,
):
    restored = restore_completed_model_result(model_name)
    if restored is not None:
        return restored

    _, P_oof = run_endpoint_cv(
        model_name,
        model_factory,
        fold_feature_builder,
    )

    P_validation = np.full(
        (len(val_idx), len(TARGETS)),
        np.nan,
        dtype=float,
    )
    P_test = np.full(
        (len(test_idx), len(TARGETS)),
        np.nan,
        dtype=float,
    )

    model_save_dir = MODEL_DIR / slugify(model_name)
    model_save_dir.mkdir(parents=True, exist_ok=True)

    progress_path = RESUME_DIR / slugify(model_name) / "final_endpoints.npz"
    progress_path.parent.mkdir(parents=True, exist_ok=True)

    completed = np.zeros(len(TARGETS), dtype=bool)
    oob_values = np.full(len(TARGETS), np.nan, dtype=float)

    if progress_path.exists():
        try:
            progress = np.load(progress_path)
            cached_validation = progress["P_validation"]
            cached_test = progress["P_test"]
            cached_completed = progress["completed"].astype(bool)
            cached_oob = progress["oob_values"]

            if (
                cached_validation.shape == P_validation.shape
                and cached_test.shape == P_test.shape
            ):
                P_validation = cached_validation
                P_test = cached_test
                completed = cached_completed
                oob_values = cached_oob
                print(
                    f"Resuming {model_name}: "
                    f"{int(completed.sum())}/{len(TARGETS)} endpoints complete."
                )
        except Exception:
            pass

    for j, target in enumerate(TARGETS):
        if completed[j]:
            continue

        y_train_endpoint = Y_train[:, j]
        train_mask = np.isfinite(y_train_endpoint)
        fallback = prior_probability(y_train_endpoint)

        if (
            train_mask.sum() < 10
            or len(np.unique(y_train_endpoint[train_mask])) < 2
        ):
            P_validation[:, j] = fallback
            P_test[:, j] = fallback
            completed[j] = True
        else:
            model = model_factory(y_train_endpoint[train_mask])
            model = fit_endpoint_estimator(
                model,
                X_train[train_mask],
                y_train_endpoint[train_mask].astype(int),
                X_validation,
                Y_val[:, j],
            )

            P_validation[:, j] = np.clip(
                predict_positive_probability(model, X_validation),
                0.0,
                1.0,
            )
            P_test[:, j] = np.clip(
                predict_positive_probability(model, X_test),
                0.0,
                1.0,
            )

            oob_values[j] = (
                float(model.oob_score_)
                if hasattr(model, "oob_score_")
                else np.nan
            )

            joblib.dump(
                model,
                model_save_dir / f"{j:02d}_{slugify(target)}.joblib",
            )
            completed[j] = True

        ultra_atomic_save_npz(
            progress_path,
            P_validation=P_validation,
            P_test=P_test,
            completed=completed,
            oob_values=oob_values,
        )

    oob_table = pd.DataFrame({
        "Endpoint": TARGETS,
        "OOB Accuracy": oob_values,
    })

    if oob_table["OOB Accuracy"].notna().any():
        oob_table.to_csv(
            model_save_dir / "oob_accuracy_by_endpoint.csv",
            index=False,
        )

    return register_model_result(
        model_name,
        P_oof,
        P_validation,
        P_test,
        diagnostic_method="3-fold out-of-fold train prediction",
        notes=(
            "Endpoint-wise regularized model. Final estimators were fitted "
            "on the 70% training split only. Test labels were not used. "
            "Endpoint progress is checkpointed after every completed task."
        ),
    )

# Classical Machine-Learning Models

The first four models use molecular fingerprints and RDKit descriptors. Random Forest and Extra Trees are bagging-based methods, while XGBoost is a boosting method.

## Model 1 — Regularized Random Forest

Key controls: limited tree depth, minimum leaf size, feature subsampling, row subsampling and balanced bootstrap learning.

In [ ]:
def random_forest_factory(y):
    return RandomForestClassifier(
        n_estimators=900,
        max_depth=18,
        min_samples_split=8,
        min_samples_leaf=4,
        max_features=0.25,
        max_samples=0.85,
        class_weight="balanced_subsample",
        bootstrap=True,
        oob_score=True,
        random_state=SEED,
        n_jobs=N_JOBS,
    )

In [ ]:
rf_summary, rf_per_endpoint = run_classical_endpoint_model(
    "Regularized Random Forest",
    random_forest_factory,
    X_train_chem,
    X_val_chem,
    X_test_chem,
    build_chemical_features_for_fold,
)

## Model 2 — Regularized Extra Trees

Extra Trees adds stronger split randomness than Random Forest. This increases tree diversity and can reduce variance.

In [ ]:
def extra_trees_factory(y):
    return ExtraTreesClassifier(
        n_estimators=1000,
        max_depth=20,
        min_samples_split=8,
        min_samples_leaf=4,
        max_features=0.35,
        bootstrap=True,
        oob_score=True,
        max_samples=0.90,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=N_JOBS,
    )

In [ ]:
et_summary, et_per_endpoint = run_classical_endpoint_model(
    "Regularized Extra Trees",
    extra_trees_factory,
    X_train_chem,
    X_val_chem,
    X_test_chem,
    build_chemical_features_for_fold,
)

## Model 3 — Regularized XGBoost

XGBoost builds trees sequentially. Each new tree focuses on errors left by previous trees. Strong row/feature subsampling, shallow trees, L1/L2 penalties and validation early stopping are used to control overfitting.

In [ ]:
def xgboost_factory(y):
    positive_count = max(int(np.sum(y == 1)), 1)
    negative_count = max(int(np.sum(y == 0)), 1)

    return XGBClassifier(
        n_estimators=2500,
        max_depth=3,
        min_child_weight=6,
        learning_rate=0.015,
        subsample=0.75,
        colsample_bytree=0.70,
        gamma=0.15,
        reg_alpha=0.10,
        reg_lambda=7.0,
        max_delta_step=1,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=min(negative_count / positive_count, 20.0),
        early_stopping_rounds=80,
        random_state=SEED,
        n_jobs=N_JOBS,
        tree_method="hist",
    )

In [ ]:
xgb_summary, xgb_per_endpoint = run_classical_endpoint_model(
    "Regularized XGBoost",
    xgboost_factory,
    X_train_chem,
    X_val_chem,
    X_test_chem,
    build_chemical_features_for_fold,
)

## Optional robustness experiment — Scaffold-group cross-validation

This section implements the highest-priority scientific recommendation from the technical review.

When `RUN_SCAFFOLD_GROUP_CV = True`, Random Forest, Extra Trees and XGBoost are evaluated with 3-fold `GroupKFold` using Bemis-Murcko scaffold IDs. All descriptor preprocessing is refitted inside each fold.

Only ROC-AUC and Accuracy are reported.

The section is disabled by default because it adds many endpoint-specific model fits. Enable it for the final research run when sufficient CPU time is available.

In [ ]:
def run_scaffold_group_cv(
    model_name,
    model_factory,
):
    scaffold_groups = SCAFFOLD_IDS[train_idx]

    splitter = GroupKFold(
        n_splits=CV_FOLDS
    )

    fold_rows = []

    for fold_number, (
        train_relative,
        validation_relative,
    ) in enumerate(
        splitter.split(
            np.arange(len(train_idx)),
            groups=scaffold_groups,
        ),
        start=1,
    ):
        train_absolute = train_idx[
            train_relative
        ]

        validation_absolute = train_idx[
            validation_relative
        ]

        (
            X_fold_train,
            X_fold_validation,
        ) = build_chemical_features_for_fold(
            train_absolute,
            validation_absolute,
        )

        Y_fold_train = Y[
            train_absolute
        ]

        Y_fold_validation = Y[
            validation_absolute
        ]

        P_fold_validation = np.full(
            (
                len(validation_relative),
                len(TARGETS),
            ),
            np.nan,
            dtype=float,
        )

        for j, target in enumerate(TARGETS):
            y_train_endpoint = (
                Y_fold_train[:, j]
            )

            train_mask = np.isfinite(
                y_train_endpoint
            )

            fallback = prior_probability(
                y_train_endpoint
            )

            if (
                train_mask.sum() < 10
                or len(
                    np.unique(
                        y_train_endpoint[
                            train_mask
                        ]
                    )
                ) < 2
            ):
                P_fold_validation[
                    :,
                    j,
                ] = fallback

                continue

            model = model_factory(
                y_train_endpoint[
                    train_mask
                ]
            )

            model = fit_endpoint_estimator(
                model,
                X_fold_train[
                    train_mask
                ],
                y_train_endpoint[
                    train_mask
                ].astype(int),
                X_fold_validation,
                Y_fold_validation[:, j],
            )

            P_fold_validation[:, j] = (
                np.clip(
                    predict_positive_probability(
                        model,
                        X_fold_validation,
                    ),
                    0.0,
                    1.0,
                )
            )

        fold_auc = mean_metric([
            safe_auc(
                Y_fold_validation[:, j],
                P_fold_validation[:, j],
            )
            for j in range(
                len(TARGETS)
            )
        ])

        fold_accuracy = mean_metric([
            safe_accuracy(
                Y_fold_validation[:, j],
                P_fold_validation[:, j],
                THRESHOLD,
            )
            for j in range(
                len(TARGETS)
            )
        ])

        fold_rows.append({
            "Model": model_name,
            "Fold": fold_number,
            "Scaffold CV Mean AUC": (
                fold_auc
            ),
            "Scaffold CV Mean Accuracy": (
                fold_accuracy
            ),
            "Validation Compounds": int(
                len(validation_relative)
            ),
            "Validation Scaffolds": int(
                len(
                    set(
                        scaffold_groups[
                            validation_relative
                        ]
                    )
                )
            ),
        })

        print(
            f"{model_name} | Scaffold fold "
            f"{fold_number}: "
            f"AUC={fold_auc:.4f}, "
            f"Accuracy={fold_accuracy:.4f}"
        )

    return pd.DataFrame(
        fold_rows
    )


if RUN_SCAFFOLD_GROUP_CV:
    scaffold_cv_tables = []

    for model_name, model_factory in [
        (
            "Regularized Random Forest",
            random_forest_factory,
        ),
        (
            "Regularized Extra Trees",
            extra_trees_factory,
        ),
        (
            "Regularized XGBoost",
            xgboost_factory,
        ),
    ]:
        scaffold_cv_tables.append(
            run_scaffold_group_cv(
                model_name,
                model_factory,
            )
        )

    scaffold_cv_results = pd.concat(
        scaffold_cv_tables,
        ignore_index=True,
    )

    scaffold_cv_summary = (
        scaffold_cv_results
        .groupby("Model")
        .agg(
            Scaffold_CV_Mean_AUC=(
                "Scaffold CV Mean AUC",
                "mean",
            ),
            Scaffold_CV_SD_AUC=(
                "Scaffold CV Mean AUC",
                "std",
            ),
            Scaffold_CV_Mean_Accuracy=(
                "Scaffold CV Mean Accuracy",
                "mean",
            ),
        )
        .reset_index()
    )

    scaffold_cv_results.to_csv(
        OUTPUT_DIR
        / "scaffold_group_cv_folds.csv",
        index=False,
    )

    scaffold_cv_summary.to_csv(
        OUTPUT_DIR
        / "scaffold_group_cv_summary.csv",
        index=False,
    )

    display(
        scaffold_cv_summary.style
        .format({
            "Scaffold_CV_Mean_AUC": "{:.4f}",
            "Scaffold_CV_SD_AUC": "{:.4f}",
            "Scaffold_CV_Mean_Accuracy": "{:.4f}",
        })
        .set_caption(
            "Scaffold-Group CV — ROC-AUC and Accuracy"
        )
    )
else:
    print(
        "Optional scaffold-group CV is disabled. "
        "Set RUN_SCAFFOLD_GROUP_CV = True "
        "for the final robustness experiment."
    )

## Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel

The SVM uses exact binary Tanimoto similarity between Morgan ECFP4 fingerprints. The regularization parameter `C` is selected by train-only cross-validation with the one-standard-error rule, which prefers a lower-complexity value when performance is statistically similar.

In [ ]:
def compute_tanimoto_kernel(
    A,
    B,
    block_size=256,
    cache_path=None,
    progress_path=None,
):
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)
    expected_shape = (len(A), len(B))

    if cache_path is None:
        kernel = np.empty(expected_shape, dtype=np.float32)
        start_row = 0
    else:
        cache_path = Path(cache_path)
        progress_path = Path(progress_path)
        cache_path.parent.mkdir(parents=True, exist_ok=True)

        progress = _ultra_read_json(progress_path, {})

        if cache_path.exists() and progress.get("completed") is True:
            cached = np.load(cache_path, mmap_mode="r")
            if cached.shape == expected_shape:
                print(f"Restored Tanimoto kernel: {cache_path.name}")
                return cached

        if cache_path.exists():
            try:
                kernel = np.lib.format.open_memmap(
                    cache_path,
                    mode="r+",
                    dtype="float32",
                    shape=expected_shape,
                )
            except Exception:
                cache_path.unlink(missing_ok=True)
                kernel = np.lib.format.open_memmap(
                    cache_path,
                    mode="w+",
                    dtype="float32",
                    shape=expected_shape,
                )
        else:
            kernel = np.lib.format.open_memmap(
                cache_path,
                mode="w+",
                dtype="float32",
                shape=expected_shape,
            )

        start_row = int(progress.get("completed_rows", 0))
        start_row = max(0, min(start_row, len(A)))

    B_sum = B.sum(axis=1)[None, :]

    for start in tqdm(
        range(start_row, len(A), block_size),
        desc="Computing Tanimoto kernel",
        initial=start_row,
        total=len(A),
    ):
        end = min(start + block_size, len(A))
        block = A[start:end]
        intersection = block @ B.T
        denominator = (
            block.sum(axis=1)[:, None]
            + B_sum
            - intersection
        )

        kernel[start:end] = np.divide(
            intersection,
            denominator,
            out=np.zeros_like(intersection),
            where=denominator > 0,
        )

        if cache_path is not None:
            kernel.flush()
            _ultra_atomic_write_json(
                progress_path,
                {
                    "completed_rows": end,
                    "total_rows": len(A),
                    "completed": end == len(A),
                },
            )

    if cache_path is not None:
        kernel.flush()
        return np.load(cache_path, mmap_mode="r")

    return kernel

In [ ]:
X_train_ecfp = X_morgan[train_idx].astype(np.float32)
X_val_ecfp = X_morgan[val_idx].astype(np.float32)
X_test_ecfp = X_morgan[test_idx].astype(np.float32)

K_train = compute_tanimoto_kernel(
    X_train_ecfp,
    X_train_ecfp,
    cache_path=FEATURE_DIR / "tanimoto_train.npy",
    progress_path=RESUME_DIR / "tanimoto_train_progress.json",
)
K_validation = compute_tanimoto_kernel(
    X_val_ecfp,
    X_train_ecfp,
    cache_path=FEATURE_DIR / "tanimoto_validation.npy",
    progress_path=RESUME_DIR / "tanimoto_validation_progress.json",
)
K_test = compute_tanimoto_kernel(
    X_test_ecfp,
    X_train_ecfp,
    cache_path=FEATURE_DIR / "tanimoto_test.npy",
    progress_path=RESUME_DIR / "tanimoto_test_progress.json",
)

print("Train kernel shape:", K_train.shape)
print("Validation kernel shape:", K_validation.shape)
print("Test kernel shape:", K_test.shape)

In [ ]:
def select_tanimoto_c(K, y, c_grid=None):
    if c_grid is None:
        c_grid = (
            [0.10, 0.25, 0.50, 1.0, 2.0]
            if RESOURCE_MODE == "low"
            else [0.05, 0.10, 0.25, 0.50, 1.0, 2.0, 4.0, 8.0]
        )

    K = np.asarray(K)
    y = np.asarray(y).astype(int)

    splitter = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED,
    )

    rows = []

    for c_value in c_grid:
        fold_scores = []

        for train_relative, validation_relative in splitter.split(
            np.arange(len(y)),
            y,
        ):
            model = SVC(
                kernel="precomputed",
                C=float(c_value),
                class_weight="balanced",
                probability=False,
                cache_size=1500,
            )

            model.fit(
                K[np.ix_(train_relative, train_relative)],
                y[train_relative],
            )

            score = model.decision_function(
                K[np.ix_(validation_relative, train_relative)]
            )

            fold_scores.append(
                safe_auc(y[validation_relative], score)
            )

        rows.append({
            "C": float(c_value),
            "Mean AUC": mean_metric(fold_scores),
            "Std AUC": float(np.nanstd(fold_scores, ddof=1)),
        })

    cv_table = pd.DataFrame(rows).sort_values("C").reset_index(drop=True)

    best_position = int(cv_table["Mean AUC"].to_numpy().argmax())
    best_row = cv_table.iloc[best_position]
    best_mean = float(best_row["Mean AUC"])
    best_standard_error = float(best_row["Std AUC"]) / math.sqrt(3)
    cutoff = best_mean - best_standard_error

    eligible = cv_table[cv_table["Mean AUC"] >= cutoff]
    selected = eligible.sort_values("C").iloc[0]

    return float(selected["C"]), float(selected["Mean AUC"]), cv_table

In [ ]:
def tanimoto_oof_predictions(K_train, y_full, selected_c):
    y_full = np.asarray(y_full, dtype=float)
    labeled_indices = np.where(np.isfinite(y_full))[0]
    labeled_y = y_full[labeled_indices].astype(int)

    fallback = prior_probability(y_full)
    prediction = np.full(len(y_full), fallback, dtype=float)

    if len(labeled_y) < 15 or len(np.unique(labeled_y)) < 2:
        return prediction

    splitter = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED,
    )

    for train_relative, validation_relative in splitter.split(
        np.arange(len(labeled_y)),
        labeled_y,
    ):
        train_positions = labeled_indices[train_relative]
        validation_positions = labeled_indices[validation_relative]

        model = SVC(
            kernel="precomputed",
            C=selected_c,
            class_weight="balanced",
            probability=False,
            cache_size=1500,
        )

        model.fit(
            K_train[np.ix_(train_positions, train_positions)],
            y_full[train_positions].astype(int),
        )

        score = model.decision_function(
            K_train[np.ix_(validation_positions, train_positions)]
        )

        prediction[validation_positions] = 1.0 / (
            1.0 + np.exp(-np.clip(score, -50, 50))
        )

    return prediction

In [ ]:
def run_tanimoto_svm(model_name, K_train, K_validation, K_test):
    restored = restore_completed_model_result(model_name)
    if restored is not None:
        selected_path = OUTPUT_DIR / "svm_selected_c.csv"
        selected_rows = (
            pd.read_csv(selected_path).to_dict("records")
            if selected_path.exists()
            else []
        )
        return restored, {
            row["Endpoint"]: row["C"]
            for row in selected_rows
        }

    P_oof = np.full((len(train_idx), len(TARGETS)), np.nan, dtype=float)
    P_validation = np.full((len(val_idx), len(TARGETS)), np.nan, dtype=float)
    P_test = np.full((len(test_idx), len(TARGETS)), np.nan, dtype=float)

    selected_c = np.full(len(TARGETS), np.nan, dtype=float)
    selected_cv_auc = np.full(len(TARGETS), np.nan, dtype=float)
    completed = np.zeros(len(TARGETS), dtype=bool)

    model_save_dir = MODEL_DIR / slugify(model_name)
    model_save_dir.mkdir(parents=True, exist_ok=True)

    progress_path = RESUME_DIR / slugify(model_name) / "endpoint_progress.npz"
    progress_path.parent.mkdir(parents=True, exist_ok=True)

    if progress_path.exists():
        try:
            progress = np.load(progress_path)
            P_oof = progress["P_oof"]
            P_validation = progress["P_validation"]
            P_test = progress["P_test"]
            selected_c = progress["selected_c"]
            selected_cv_auc = progress["selected_cv_auc"]
            completed = progress["completed"].astype(bool)
            print(
                f"Resuming {model_name}: "
                f"{int(completed.sum())}/{len(TARGETS)} endpoints complete."
            )
        except Exception:
            pass

    for j, target in enumerate(TARGETS):
        if completed[j]:
            continue

        y_train_endpoint = Y_train[:, j]
        labeled_indices = np.where(np.isfinite(y_train_endpoint))[0]
        labeled_y = y_train_endpoint[labeled_indices].astype(int)
        fallback = prior_probability(y_train_endpoint)

        if len(labeled_y) < 15 or len(np.unique(labeled_y)) < 2:
            P_oof[:, j] = fallback
            P_validation[:, j] = fallback
            P_test[:, j] = fallback
            selected_c[j] = 0.5
            completed[j] = True
        else:
            labeled_kernel = K_train[np.ix_(labeled_indices, labeled_indices)]
            best_c, cv_auc, cv_table = select_tanimoto_c(
                labeled_kernel,
                labeled_y,
            )

            cv_table.to_csv(
                OUTPUT_DIR / f"svm_{slugify(target)}_c_search.csv",
                index=False,
            )

            P_oof[:, j] = tanimoto_oof_predictions(
                K_train,
                y_train_endpoint,
                best_c,
            )

            model = SVC(
                kernel="precomputed",
                C=best_c,
                class_weight="balanced",
                probability=False,
                cache_size=1500,
                random_state=SEED,
            )

            model.fit(
                K_train[np.ix_(labeled_indices, labeled_indices)],
                labeled_y,
            )

            validation_score = model.decision_function(
                K_validation[:, labeled_indices]
            )
            test_score = model.decision_function(
                K_test[:, labeled_indices]
            )

            P_validation[:, j] = 1.0 / (
                1.0 + np.exp(-np.clip(validation_score, -50, 50))
            )
            P_test[:, j] = 1.0 / (
                1.0 + np.exp(-np.clip(test_score, -50, 50))
            )

            joblib.dump(
                model,
                model_save_dir / f"{j:02d}_{slugify(target)}.joblib",
            )

            selected_c[j] = best_c
            selected_cv_auc[j] = cv_auc
            completed[j] = True

        ultra_atomic_save_npz(
            progress_path,
            P_oof=P_oof,
            P_validation=P_validation,
            P_test=P_test,
            selected_c=selected_c,
            selected_cv_auc=selected_cv_auc,
            completed=completed,
        )

    selected_rows = [
        {
            "Endpoint": target,
            "C": float(selected_c[j]),
            "CV AUC": float(selected_cv_auc[j]),
        }
        for j, target in enumerate(TARGETS)
    ]

    pd.DataFrame(selected_rows).to_csv(
        OUTPUT_DIR / "svm_selected_c.csv",
        index=False,
    )

    result = register_model_result(
        model_name,
        P_oof,
        P_validation,
        P_test,
        diagnostic_method="3-fold OOF Tanimoto-SVM prediction",
        notes=(
            "Exact ECFP4 Tanimoto kernel. C was selected with train-only "
            "cross-validation and the one-standard-error rule. Endpoint "
            "progress is checkpointed."
        ),
    )

    return result, {
        row["Endpoint"]: row["C"]
        for row in selected_rows
    }

In [ ]:
(
    svm_tanimoto_result,
    TANIMOTO_C_BY_ENDPOINT,
) = run_tanimoto_svm(
    "Regularized SVM Tanimoto",
    K_train,
    K_validation,
    K_test,
)

svm_tanimoto_summary, svm_tanimoto_per_endpoint = svm_tanimoto_result

# Deep-Learning Infrastructure

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.as_tensor(
            np.asarray(X),
            dtype=torch.float32,
        )
        self.Y = torch.as_tensor(
            np.asarray(Y),
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.Y[index]

In [ ]:
def compute_pos_weight(Y_matrix, cap=10.0, power=0.75):
    weights = []

    for j in range(Y_matrix.shape[1]):
        y = Y_matrix[:, j]
        positive = max(int(np.nansum(y == 1)), 1)
        negative = max(int(np.nansum(y == 0)), 1)

        ratio = (negative / positive) ** power
        weights.append(min(ratio, cap))

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    )

In [ ]:
def masked_weighted_bce(
    logits,
    targets,
    pos_weight=None,
    label_smoothing=0.01,
):
    mask = torch.isfinite(targets)
    safe_targets = torch.nan_to_num(targets, nan=0.0)

    if label_smoothing > 0:
        safe_targets = (
            safe_targets * (1.0 - label_smoothing)
            + 0.5 * label_smoothing
        )

    loss = F.binary_cross_entropy_with_logits(
        logits,
        safe_targets,
        reduction="none",
        pos_weight=pos_weight,
    )

    return (
        (loss * mask.float()).sum()
        / mask.float().sum().clamp_min(1.0)
    )

In [ ]:
def masked_focal_bce(
    logits,
    targets,
    pos_weight=None,
    gamma=1.0,
    label_smoothing=0.01,
):
    mask = torch.isfinite(targets)
    safe_targets = torch.nan_to_num(targets, nan=0.0)

    if label_smoothing > 0:
        safe_targets = (
            safe_targets * (1.0 - label_smoothing)
            + 0.5 * label_smoothing
        )

    bce = F.binary_cross_entropy_with_logits(
        logits,
        safe_targets,
        reduction="none",
        pos_weight=pos_weight,
    )

    probability = torch.sigmoid(logits)
    pt = (
        safe_targets * probability
        + (1.0 - safe_targets) * (1.0 - probability)
    )

    focal_factor = (1.0 - pt).clamp_min(1e-6).pow(gamma)
    loss = bce * focal_factor

    return (
        (loss * mask.float()).sum()
        / mask.float().sum().clamp_min(1.0)
    )

In [ ]:
def filter_sparse_binary_features(
    X_train,
    X_validation,
    X_test,
    binary_dimension,
    min_count=8,
    max_fraction=0.995,
    audit_label="Sparse binary features",
):
    X_train = np.asarray(X_train)
    X_validation = np.asarray(X_validation)
    X_test = np.asarray(X_test)

    binary_train = X_train[:, :binary_dimension]
    count = np.sum(binary_train > 0.5, axis=0)

    keep = (
        (count >= min_count)
        & (count <= int(len(X_train) * max_fraction))
    )

    X_train_new = np.hstack([
        X_train[:, :binary_dimension][:, keep],
        X_train[:, binary_dimension:],
    ])

    X_validation_new = np.hstack([
        X_validation[:, :binary_dimension][:, keep],
        X_validation[:, binary_dimension:],
    ])

    X_test_new = np.hstack([
        X_test[:, :binary_dimension][:, keep],
        X_test[:, binary_dimension:],
    ])

    retained_count = int(keep.sum())

    SPARSE_FEATURE_AUDIT.append({
        "Feature Set": audit_label,
        "Input Binary Features": int(binary_dimension),
        "Retained Binary Features": retained_count,
        "Removed Binary Features": int(
            binary_dimension - retained_count
        ),
        "Minimum Count": int(min_count),
        "Maximum Fraction": float(max_fraction),
    })

    print(
        f"{audit_label}: "
        f"{binary_dimension} -> {retained_count} retained"
    )

    return (
        X_train_new.astype(np.float32),
        X_validation_new.astype(np.float32),
        X_test_new.astype(np.float32),
        keep,
    )

In [ ]:
@torch.no_grad()
def predict_tabular_torch(
    model,
    X,
    batch_size=512,
    probability_function=None,
):
    model.eval()

    loader = DataLoader(
        torch.as_tensor(np.asarray(X), dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False,
    )

    predictions = []

    for X_batch in loader:
        raw_output = model(X_batch.to(DEVICE))

        if probability_function is None:
            probability = torch.sigmoid(raw_output)
        else:
            probability = probability_function(raw_output)

        predictions.append(
            probability.detach().cpu().numpy()
        )

    return np.vstack(predictions)

In [ ]:
def clone_state_dict(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def average_state_dicts(state_dicts):
    if len(state_dicts) == 1:
        return state_dicts[0]

    averaged = {}

    for key in state_dicts[0]:
        values = [state[key] for state in state_dicts]

        if torch.is_floating_point(values[0]):
            averaged[key] = torch.stack(values, dim=0).mean(dim=0)
        else:
            averaged[key] = values[0]

    return averaged

In [ ]:
def train_tabular_multitask(
    model_name,
    model,
    X_train,
    Y_train_local,
    X_validation,
    Y_validation_local,
    X_test,
    optimizer,
    max_epochs,
    batch_size,
    patience,
    loss_function,
    scheduler=None,
    probability_function=None,
    notes="",
    top_k=3,
    min_delta=2e-4,
    minimum_epochs=10,
):
    restored = restore_completed_model_result(model_name)
    if restored is not None:
        return restored

    train_loader = DataLoader(
        TabularDataset(X_train, Y_train_local),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

    validation_loader = DataLoader(
        TabularDataset(X_validation, Y_validation_local),
        batch_size=max(batch_size, 256),
        shuffle=False,
        num_workers=0,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": [],
        "learning_rate": [],
    }

    best_validation_auc = -np.inf
    wait = 0
    top_states = []
    start_epoch = 0

    model = model.to(DEVICE)

    checkpoint_dir = RESUME_DIR / slugify(model_name)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "training_checkpoint.pt"

    if checkpoint_path.exists():
        try:
            checkpoint = torch.load(
                checkpoint_path,
                map_location=DEVICE,
                weights_only=False,
            )
            model.load_state_dict(checkpoint["model_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])

            if scheduler is not None and checkpoint.get("scheduler_state"):
                scheduler.load_state_dict(checkpoint["scheduler_state"])

            history = checkpoint["history"]
            best_validation_auc = float(checkpoint["best_validation_auc"])
            wait = int(checkpoint["wait"])
            top_states = checkpoint["top_states"]
            start_epoch = int(checkpoint["next_epoch"])

            print(
                f"Resuming {model_name} from epoch {start_epoch + 1}."
            )
        except Exception as exc:
            print(
                f"Training checkpoint for {model_name} could not be restored; "
                f"training will restart. Details: {exc}"
            )
            history = {
                "train_loss": [],
                "val_loss": [],
                "val_auc": [],
                "learning_rate": [],
            }
            best_validation_auc = -np.inf
            wait = 0
            top_states = []
            start_epoch = 0

    for epoch in range(start_epoch, max_epochs):
        model.train()
        train_losses = []

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            Y_batch = Y_batch.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            raw_output = model(X_batch)
            loss = loss_function(raw_output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )

            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        validation_losses = []
        validation_probabilities = []
        validation_targets = []

        with torch.no_grad():
            for X_batch, Y_batch in validation_loader:
                X_batch = X_batch.to(DEVICE)
                Y_batch = Y_batch.to(DEVICE)

                raw_output = model(X_batch)
                validation_losses.append(
                    float(loss_function(raw_output, Y_batch).detach().cpu())
                )

                probability = (
                    torch.sigmoid(raw_output)
                    if probability_function is None
                    else probability_function(raw_output)
                )

                validation_probabilities.append(probability.cpu().numpy())
                validation_targets.append(Y_batch.cpu().numpy())

        train_loss = float(np.mean(train_losses))
        validation_loss = float(np.mean(validation_losses))
        validation_probability = np.vstack(validation_probabilities)
        validation_target = np.vstack(validation_targets)

        validation_auc = mean_metric([
            safe_auc(
                validation_target[:, j],
                validation_probability[:, j],
            )
            for j in range(len(TARGETS))
        ])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(validation_loss)
        history["val_auc"].append(validation_auc)
        history["learning_rate"].append(
            float(optimizer.param_groups[0]["lr"])
        )

        if scheduler is not None:
            try:
                scheduler.step(validation_auc)
            except TypeError:
                scheduler.step()

        score = validation_auc if np.isfinite(validation_auc) else -np.inf

        top_states.append({
            "epoch": epoch + 1,
            "val_auc": float(score),
            "state_dict": clone_state_dict(model),
        })
        top_states = sorted(
            top_states,
            key=lambda item: item["val_auc"],
            reverse=True,
        )[:top_k]

        if score > best_validation_auc + min_delta:
            best_validation_auc = score
            wait = 0
        else:
            wait += 1

        ultra_atomic_torch_save(
            {
                "model_state": clone_state_dict(model),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": (
                    scheduler.state_dict() if scheduler is not None else None
                ),
                "history": history,
                "best_validation_auc": best_validation_auc,
                "wait": wait,
                "top_states": top_states,
                "next_epoch": epoch + 1,
            },
            checkpoint_path,
        )

        print(
            f"Epoch {epoch + 1:03d}/{max_epochs} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={validation_loss:.5f} | "
            f"val_auc={validation_auc:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if epoch + 1 >= minimum_epochs and wait >= patience:
            print(
                f"Validation-AUC early stopping after {patience} "
                "epochs without sufficient improvement."
            )
            break

    valid_top_states = [
        item for item in top_states
        if np.isfinite(item["val_auc"])
    ]

    if valid_top_states:
        averaged_state = average_state_dicts([
            item["state_dict"]
            for item in valid_top_states
        ])
        model.load_state_dict(averaged_state)

        print(
            "Top checkpoint epochs:",
            [
                (item["epoch"], round(item["val_auc"], 4))
                for item in valid_top_states
            ],
        )

    model = model.to(DEVICE)

    P_train = predict_tabular_torch(
        model,
        X_train,
        probability_function=probability_function,
    )
    P_validation = predict_tabular_torch(
        model,
        X_validation,
        probability_function=probability_function,
    )
    P_test = predict_tabular_torch(
        model,
        X_test,
        probability_function=probability_function,
    )

    history["best_val_auc"] = float(best_validation_auc)
    history["top_checkpoint_records"] = [
        {
            "epoch": item["epoch"],
            "val_auc": item["val_auc"],
        }
        for item in valid_top_states
    ]

    model_path = MODEL_DIR / f"{slugify(model_name)}.pt"
    torch.save(model.state_dict(), model_path)

    result = register_model_result(
        model_name,
        P_train,
        P_validation,
        P_test,
        history=history,
        diagnostic_method=(
            "Best-checkpoint train prediction; model selected by validation "
            "AUC and top-checkpoint weight averaging"
        ),
        notes=notes,
    )

    checkpoint_path.unlink(missing_ok=True)

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result

## Model 5 — Strongly Regularized DeepTox-style Multi-task DNN

The shared network is smaller than the original implementation and now uses task-specific output heads.

Architecture:

`Input -> 512 -> 256 -> 128 shared representation -> twelve 32-unit task heads -> 12 logits`

This keeps multi-task feature sharing while allowing each toxicity endpoint to learn a small specialized decision boundary. Strong dropout, weight decay, sparse-feature filtering, masked focal BCE, validation-AUC early stopping and top-checkpoint averaging are retained.

In [ ]:
(
    X_train_deeptox_raw,
    X_val_deeptox_raw,
    X_test_deeptox_raw,
    DEEPTOX_KEEP_MASK,
) = filter_sparse_binary_features(
    X_train_chem,
    X_val_chem,
    X_test_chem,
    binary_dimension=N_BITS + 167,
    min_count=8,
    max_fraction=0.995,
    audit_label="DeepTox ECFP4 + MACCS",
)

X_train_deeptox = np.tanh(X_train_deeptox_raw).astype(np.float32)
X_val_deeptox = np.tanh(X_val_deeptox_raw).astype(np.float32)
X_test_deeptox = np.tanh(X_test_deeptox_raw).astype(np.float32)

np.save(
    MODEL_DIR / "deeptox_sparse_binary_keep_mask.npy",
    DEEPTOX_KEEP_MASK,
)

In [ ]:
class DeepToxMTDNN(nn.Module):
    def __init__(
        self,
        input_dimension,
        number_of_tasks,
    ):
        super().__init__()

        self.input_dropout = nn.Dropout(0.15)

        self.shared_trunk = nn.Sequential(
            nn.Linear(input_dimension, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.45),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.40),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.35),
        )

        self.task_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(128, 32),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(32, 1),
            )
            for _ in range(number_of_tasks)
        ])

    def forward(self, X):
        shared = self.shared_trunk(
            self.input_dropout(X)
        )

        logits = [
            head(shared)
            for head in self.task_heads
        ]

        return torch.cat(logits, dim=1)

In [ ]:
deeptox_model = DeepToxMTDNN(
    X_train_deeptox.shape[1],
    len(TARGETS),
).to(DEVICE)

pos_weight_deeptox = compute_pos_weight(
    Y_train,
    cap=8.0,
    power=0.70,
)

deeptox_optimizer = torch.optim.AdamW(
    deeptox_model.parameters(),
    lr=2.5e-4,
    weight_decay=5e-4,
)

deeptox_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    deeptox_optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
)

In [ ]:
deeptox_summary, deeptox_per_endpoint = train_tabular_multitask(
    "Strongly Regularized DeepTox-style MT-DNN",
    deeptox_model,
    X_train_deeptox,
    Y_train,
    X_val_deeptox,
    Y_val,
    X_test_deeptox,
    optimizer=deeptox_optimizer,
    max_epochs=160,
    batch_size=512 if RESOURCE_MODE != "low" else 256,
    patience=18,
    minimum_epochs=15,
    loss_function=lambda raw, y: masked_focal_bce(
        raw,
        y,
        pos_weight=pos_weight_deeptox,
        gamma=0.75,
        label_smoothing=0.015,
    ),
    scheduler=deeptox_scheduler,
    top_k=3,
    notes=(
        "Reduced 512-256-128 shared trunk with twelve small task-specific heads; "
        "ECFP4, MACCS and filtered RDKit descriptors; strict sparse-feature "
        "filtering, tanh transformation, LayerNorm, dropout, stronger AdamW "
        "weight decay, masked focal BCE, validation-AUC early stopping and "
        "in-memory checkpoint averaging."
    ),
)

## Model 6 — Multitask CapsNet with Denoising Autoencoder Pretraining and Dynamic Routing

The previous notebook used Bernoulli RBMs on a mixed binary-and-continuous input. The technical review identified a distribution mismatch because BernoulliRBM is designed for binary or probability-like visible units.

This version therefore uses a **two-layer denoising autoencoder** by default:

`179 features -> 256 -> 128 -> reconstruction`

The pretrained 256/128 encoder weights initialize the CapsNet feature layers. Dynamic routing and the imbalance-aware focal BCE objective are retained.

The 13-property vector also removes the duplicated LogP-as-LogD proxy and replaces it with molecular refractivity (`MolMR`).

In [ ]:
def esol_estimate(mol):
    logp = Crippen.MolLogP(mol)
    molecular_weight = Descriptors.MolWt(mol)
    rotatable_bonds = Lipinski.NumRotatableBonds(mol)
    heavy_atoms = max(mol.GetNumHeavyAtoms(), 1)

    aromatic_atoms = sum(
        atom.GetIsAromatic()
        for atom in mol.GetAtoms()
    )

    aromatic_fraction = aromatic_atoms / heavy_atoms

    return (
        0.16
        - 1.5 * logp
        - 0.01 * (molecular_weight - 40.0)
        + 0.066 * rotatable_bonds
        + 0.066 * aromatic_fraction
    )


CAPSNET_DESCRIPTOR_NAMES = [
    "MolLogP",
    "MolMR",
    "ESOL_Estimate",
    "MolWt",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "AromaticRingCount",
    "NitrogenOxygenCount",
    "TPSA",
    "FractionalPolarSurfaceArea",
    "LabuteASA",
]


def capsnet_13_descriptors(mol):
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    labute_area = rdMolDescriptors.CalcLabuteASA(mol)

    return [
        Crippen.MolLogP(mol),
        Crippen.MolMR(mol),
        esol_estimate(mol),
        Descriptors.MolWt(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        Lipinski.NumRotatableBonds(mol),
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        sum(
            atom.GetAtomicNum() in (7, 8)
            for atom in mol.GetAtoms()
        ),
        tpsa,
        tpsa / max(labute_area, 1e-6),
        labute_area,
    ]

In [ ]:
X_caps_all = np.vstack([
    np.concatenate([
        maccs_fp(mol)[1:],
        np.asarray(capsnet_13_descriptors(mol), dtype=np.float32),
    ])
    for mol in tqdm(df["mol"], desc="CapsNet features")
]).astype(np.float32)

caps_scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
X_train_caps = caps_scaler.fit_transform(
    X_caps_all[train_idx]
).astype(np.float32)
X_val_caps = caps_scaler.transform(
    X_caps_all[val_idx]
).astype(np.float32)
X_test_caps = caps_scaler.transform(
    X_caps_all[test_idx]
).astype(np.float32)

joblib.dump(
    caps_scaler,
    MODEL_DIR / "capsnet_minmax_scaler.joblib",
)

print("CapsNet input shape:", X_train_caps.shape)

In [ ]:
class CapsFeatureAutoencoder(nn.Module):
    def __init__(self, input_dimension):
        super().__init__()

        self.encoder1 = nn.Linear(
            input_dimension,
            256,
        )

        self.encoder2 = nn.Linear(
            256,
            128,
        )

        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dimension),
            nn.Sigmoid(),
        )

    def encode(self, X):
        hidden_1 = torch.relu(
            self.encoder1(X)
        )

        hidden_2 = torch.relu(
            self.encoder2(hidden_1)
        )

        return hidden_1, hidden_2

    def forward(self, X):
        _, hidden_2 = self.encode(X)
        return self.decoder(hidden_2)


def pretrain_caps_autoencoder(
    X_train,
    epochs,
    noise_std=0.04,
):
    model = CapsFeatureAutoencoder(
        X_train.shape[1]
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
    )

    loader = DataLoader(
        torch.as_tensor(
            X_train,
            dtype=torch.float32,
        ),
        batch_size=256,
        shuffle=True,
        num_workers=0,
    )

    model.train()

    for epoch in range(epochs):
        losses = []

        for clean_batch in loader:
            clean_batch = clean_batch.to(DEVICE)

            noisy_batch = (
                clean_batch
                + noise_std
                * torch.randn_like(clean_batch)
            ).clamp(0.0, 1.0)

            optimizer.zero_grad(set_to_none=True)

            reconstruction = model(noisy_batch)

            loss = F.mse_loss(
                reconstruction,
                clean_batch,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )

            optimizer.step()
            losses.append(
                float(loss.detach().cpu())
            )

        if (
            epoch == 0
            or (epoch + 1) % 10 == 0
            or epoch + 1 == epochs
        ):
            print(
                f"CapsNet autoencoder epoch "
                f"{epoch + 1:03d}/{epochs} | "
                f"MSE={np.mean(losses):.6f}"
            )

    torch.save(
        model.state_dict(),
        MODEL_DIR / "capsnet_denoising_autoencoder.pt",
    )

    return model


CAPS_AE_EPOCHS = (
    30
    if RESOURCE_MODE == "low"
    else 50
)

caps_autoencoder = pretrain_caps_autoencoder(
    X_train_caps,
    epochs=CAPS_AE_EPOCHS,
)

In [ ]:
def squash_capsules(values, dimension=-1, epsilon=1e-8):
    norm_squared = (
        values * values
    ).sum(dim=dimension, keepdim=True)

    scale = (
        norm_squared
        / (1.0 + norm_squared)
        / torch.sqrt(norm_squared + epsilon)
    )

    return scale * values

In [ ]:
class MultitaskCapsNetDAE(nn.Module):
    def __init__(
        self,
        input_dimension=179,
        number_of_tasks=12,
        routing_iterations=2,
        primary_capsules=16,
        primary_dimension=8,
        class_dimension=4,
    ):
        super().__init__()

        self.number_of_tasks = number_of_tasks
        self.routing_iterations = routing_iterations
        self.primary_capsules = primary_capsules
        self.primary_dimension = primary_dimension

        self.dropout = nn.Dropout(0.15)

        self.encoder1 = nn.Linear(
            input_dimension,
            256,
        )

        self.encoder2 = nn.Linear(
            256,
            128,
        )

        self.primary = nn.Linear(
            128,
            primary_capsules * primary_dimension,
        )

        self.route_weights = nn.Parameter(
            0.02 * torch.randn(
                number_of_tasks,
                primary_capsules,
                2,
                class_dimension,
                primary_dimension,
            )
        )

    def forward(self, X):
        X = torch.relu(
            self.encoder1(
                self.dropout(X)
            )
        )

        X = torch.relu(
            self.encoder2(
                self.dropout(X)
            )
        )

        primary = self.primary(X).view(
            -1,
            self.primary_capsules,
            self.primary_dimension,
        )

        primary = squash_capsules(
            primary,
            dimension=-1,
        )

        predicted_capsules = torch.einsum(
            "bni,tnodi->btnod",
            primary,
            self.route_weights,
        )

        routing_logits = torch.zeros(
            predicted_capsules.shape[:-1],
            device=X.device,
            dtype=X.dtype,
        )

        for routing_step in range(
            self.routing_iterations
        ):
            coupling = torch.softmax(
                routing_logits,
                dim=-1,
            )

            combined = (
                coupling.unsqueeze(-1)
                * predicted_capsules
            ).sum(dim=2)

            class_capsules = squash_capsules(
                combined,
                dimension=-1,
            )

            if (
                routing_step
                < self.routing_iterations - 1
            ):
                agreement = (
                    predicted_capsules
                    * class_capsules.unsqueeze(2)
                ).sum(dim=-1)

                routing_logits = (
                    routing_logits
                    + agreement
                )

        capsule_lengths = torch.linalg.vector_norm(
            class_capsules,
            dim=-1,
        )

        return 6.0 * (
            capsule_lengths[..., 1]
            - capsule_lengths[..., 0]
        )

In [ ]:
caps_model = MultitaskCapsNetDAE(
    input_dimension=X_train_caps.shape[1],
    number_of_tasks=len(TARGETS),
    routing_iterations=2,
).to(DEVICE)

with torch.no_grad():
    caps_model.encoder1.weight.copy_(
        caps_autoencoder.encoder1.weight
    )

    caps_model.encoder1.bias.copy_(
        caps_autoencoder.encoder1.bias
    )

    caps_model.encoder2.weight.copy_(
        caps_autoencoder.encoder2.weight
    )

    caps_model.encoder2.bias.copy_(
        caps_autoencoder.encoder2.bias
    )

print(
    "CapsNet encoder initialized from the "
    "denoising autoencoder."
)

In [ ]:
caps_pos_weight = compute_pos_weight(
    Y_train,
    cap=8.0,
    power=0.55,
)

caps_optimizer = torch.optim.AdamW(
    caps_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

caps_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    caps_optimizer,
    mode="max",
    factor=0.5,
    patience=7,
    min_lr=1e-6,
)

CAPS_EPOCHS = 180 if RESOURCE_MODE == "low" else 260

In [ ]:
caps_summary, caps_per_endpoint = train_tabular_multitask(
    "Regularized Multitask CapsNet + DAE",
    caps_model,
    X_train_caps,
    Y_train,
    X_val_caps,
    Y_val,
    X_test_caps,
    optimizer=caps_optimizer,
    max_epochs=CAPS_EPOCHS,
    batch_size=148,
    patience=30,
    minimum_epochs=20,
    loss_function=lambda raw, y: masked_focal_bce(
        raw,
        y,
        pos_weight=caps_pos_weight,
        gamma=1.0,
        label_smoothing=0.01,
    ),
    scheduler=caps_scheduler,
    top_k=3,
    notes=(
        "Paper-inspired CapsNet with 166 MACCS keys and 13 distinct "
        "molecular properties. A two-layer denoising autoencoder replaces "
        "BernoulliRBM pretraining for the mixed binary-continuous input. "
        "Dynamic routing, capped class weights, masked focal BCE, "
        "validation-AUC early stopping and checkpoint averaging are used."
    ),
)

## Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid

The hybrid performs feature-level fusion:

`ECFP4 + MACCS + filtered/scaled RDKit descriptors + DenseNet121 visual PCA features -> regularized XGBoost`

The final learner is a boosting model.

Updates in this version:

- PCA uses `whiten=False`.
- The number of visual components is selected by a 95% explained-variance rule and capped at 64.
- Chemical-only, visual-only and fused representations are compared in an ablation table.
- PCA, descriptor filtering and scaling are fitted only on the relevant training fold.

In [ ]:
class MoleculeImageDataset(Dataset):
    def __init__(self, smiles_list, transform):
        self.smiles_list = list(smiles_list)
        self.transform = transform

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, index):
        mol = Chem.MolFromSmiles(self.smiles_list[index])
        image = Draw.MolToImage(
            mol,
            size=(224, 224),
        ).convert("RGB")

        return self.transform(image)

In [ ]:
DENSENET_READY = False
X_dense_all = None

if not TORCHVISION_AVAILABLE:
    print("DenseNet121 hybrid skipped because torchvision is unavailable.")
else:
    try:
        dense_weights = DenseNet121_Weights.DEFAULT
        dense_transform = dense_weights.transforms()
        dense_backbone = densenet121(weights=dense_weights)
        dense_backbone.classifier = nn.Identity()
        dense_backbone = dense_backbone.to(DEVICE).eval()
        DENSENET_READY = True
        print("ImageNet-pretrained DenseNet121 loaded successfully.")

    except Exception as exc:
        print("DenseNet121 pretrained weights could not be loaded.")
        print("The hybrid model will be skipped rather than using random visual features.")
        print("Details:", exc)

In [ ]:
if DENSENET_READY:
    dense_feature_path = FEATURE_DIR / "densenet121_features_1024.npy"
    progress_path = RESUME_DIR / "densenet_feature_progress.json"
    progress = _ultra_read_json(progress_path, {})
    expected_dense_shape = (len(df), 1024)

    X_dense_all = None

    if (
        dense_feature_path.exists()
        and progress.get("completed") is True
    ):
        try:
            cached_dense = np.load(dense_feature_path, mmap_mode="r")
            if cached_dense.shape == expected_dense_shape:
                X_dense_all = cached_dense
                print("Restored DenseNet121 visual features from cache.")
        except Exception:
            X_dense_all = None

    if X_dense_all is None:
        image_dataset = MoleculeImageDataset(
            df["canonical_smiles"].tolist(),
            dense_transform,
        )

        image_loader = DataLoader(
            image_dataset,
            batch_size=BATCH_IMAGE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
        )

        completed_rows = int(progress.get("completed_rows", 0))
        completed_rows = max(0, min(completed_rows, len(df)))

        if dense_feature_path.exists() and completed_rows > 0:
            try:
                X_dense_memmap = np.lib.format.open_memmap(
                    dense_feature_path,
                    mode="r+",
                    dtype="float32",
                    shape=expected_dense_shape,
                )
            except Exception:
                dense_feature_path.unlink(missing_ok=True)
                completed_rows = 0
                X_dense_memmap = np.lib.format.open_memmap(
                    dense_feature_path,
                    mode="w+",
                    dtype="float32",
                    shape=expected_dense_shape,
                )
        else:
            X_dense_memmap = np.lib.format.open_memmap(
                dense_feature_path,
                mode="w+",
                dtype="float32",
                shape=expected_dense_shape,
            )
            completed_rows = 0

        row_cursor = 0

        with torch.inference_mode():
            for image_batch in tqdm(
                image_loader,
                desc="DenseNet121 feature extraction",
            ):
                batch_rows = len(image_batch)
                batch_start = row_cursor
                batch_end = row_cursor + batch_rows
                row_cursor = batch_end

                if batch_end <= completed_rows:
                    continue

                features = dense_backbone(
                    image_batch.to(DEVICE)
                ).cpu().numpy().astype(np.float32)

                write_start = max(batch_start, completed_rows)
                source_start = write_start - batch_start
                X_dense_memmap[write_start:batch_end] = features[source_start:]
                X_dense_memmap.flush()

                completed_rows = batch_end
                _ultra_atomic_write_json(
                    progress_path,
                    {
                        "completed_rows": completed_rows,
                        "total_rows": len(df),
                        "completed": completed_rows == len(df),
                    },
                )

        X_dense_memmap.flush()
        X_dense_all = np.load(dense_feature_path, mmap_mode="r")

    print("DenseNet feature shape:", X_dense_all.shape)

In [ ]:
VISUAL_TRANSFORM_CACHE = {}

def fit_visual_transformer(
    fit_indices,
    transform_indices,
    variance_target=0.95,
    maximum_components=64,
):
    cache_key = (
        hashlib.sha1(
            np.asarray(
                fit_indices,
                dtype=np.int64,
            ).tobytes()
        ).hexdigest(),
        hashlib.sha1(
            np.asarray(
                transform_indices,
                dtype=np.int64,
            ).tobytes()
        ).hexdigest(),
        float(variance_target),
        int(maximum_components),
    )

    if cache_key in VISUAL_TRANSFORM_CACHE:
        return VISUAL_TRANSFORM_CACHE[
            cache_key
        ]

    maximum_allowed = min(
        maximum_components,
        len(fit_indices) - 1,
        X_dense_all.shape[1],
    )

    variance_probe = PCA(
        n_components=variance_target,
        whiten=False,
        svd_solver="full",
    )

    variance_probe.fit(
        X_dense_all[fit_indices]
    )

    selected_components = min(
        maximum_allowed,
        int(variance_probe.n_components_),
    )

    pca = PCA(
        n_components=selected_components,
        random_state=SEED,
        svd_solver="randomized",
        whiten=False,
    )

    Z_fit = pca.fit_transform(
        X_dense_all[fit_indices]
    )

    visual_scaler = StandardScaler()

    Z_fit = visual_scaler.fit_transform(
        Z_fit
    )

    Z_transform = visual_scaler.transform(
        pca.transform(
            X_dense_all[transform_indices]
        )
    )

    explained_variance = float(
        np.sum(pca.explained_variance_ratio_)
    )

    print(
        "DenseNet PCA components:",
        selected_components,
        "| explained variance:",
        f"{explained_variance:.4f}",
    )

    return (
        Z_fit.astype(np.float32),
        Z_transform.astype(np.float32),
        pca,
        visual_scaler,
    )

In [ ]:
def build_hybrid_features_for_fold(
    fit_indices,
    transform_indices,
):
    chemical_fit, chemical_transform = (
        build_chemical_features_for_fold(
            fit_indices,
            transform_indices,
        )
    )

    visual_fit, visual_transform, _, _ = (
        fit_visual_transformer(
            fit_indices,
            transform_indices,
        )
    )

    X_fit = np.hstack([
        chemical_fit,
        visual_fit,
    ]).astype(np.float32)

    X_transform = np.hstack([
        chemical_transform,
        visual_transform,
    ]).astype(np.float32)

    return X_fit, X_transform


def build_visual_features_for_fold(
    fit_indices,
    transform_indices,
):
    visual_fit, visual_transform, _, _ = (
        fit_visual_transformer(
            fit_indices,
            transform_indices,
        )
    )

    return visual_fit, visual_transform

In [ ]:
if DENSENET_READY:
    Z_train, Z_val, dense_pca, dense_scaler = fit_visual_transformer(
        train_idx,
        val_idx,
    )

    Z_test = dense_scaler.transform(
        dense_pca.transform(X_dense_all[test_idx])
    ).astype(np.float32)

    X_train_hybrid = np.hstack([
        X_train_chem,
        Z_train,
    ]).astype(np.float32)

    X_val_hybrid = np.hstack([
        X_val_chem,
        Z_val,
    ]).astype(np.float32)

    X_test_hybrid = np.hstack([
        X_test_chem,
        Z_test,
    ]).astype(np.float32)

    joblib.dump(
        {
            "pca": dense_pca,
            "scaler": dense_scaler,
        },
        MODEL_DIR / "densenet_visual_transformer.joblib",
    )

    print("Hybrid feature shape:", X_train_hybrid.shape)

In [ ]:
def hybrid_xgboost_factory(y):
    positive_count = max(int(np.sum(y == 1)), 1)
    negative_count = max(int(np.sum(y == 0)), 1)

    return XGBClassifier(
        n_estimators=3000,
        max_depth=2,
        min_child_weight=8,
        learning_rate=0.015,
        subsample=0.75,
        colsample_bytree=0.65,
        gamma=0.20,
        reg_alpha=0.15,
        reg_lambda=10.0,
        max_delta_step=1,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=min(negative_count / positive_count, 15.0),
        early_stopping_rounds=100,
        random_state=SEED,
        n_jobs=N_JOBS,
        tree_method="hist",
    )

In [ ]:
if DENSENET_READY:
    hybrid_summary, hybrid_per_endpoint = (
        run_classical_endpoint_model(
            "Boosted DenseNet121 + Chemical XGBoost Hybrid",
            hybrid_xgboost_factory,
            X_train_hybrid,
            X_val_hybrid,
            X_test_hybrid,
            build_hybrid_features_for_fold,
        )
    )

    if RUN_HYBRID_ABLATION:
        (
            visual_only_summary,
            visual_only_per_endpoint,
        ) = run_classical_endpoint_model(
            "DenseNet121 Visual-only XGBoost Ablation",
            hybrid_xgboost_factory,
            Z_train,
            Z_val,
            Z_test,
            build_visual_features_for_fold,
        )

        hybrid_ablation_rows = []

        for model_name in [
            "Regularized XGBoost",
            "DenseNet121 Visual-only XGBoost Ablation",
            "Boosted DenseNet121 + Chemical XGBoost Hybrid",
        ]:
            if model_name in MODEL_RESULTS:
                hybrid_ablation_rows.append({
                    "Representation": model_name,
                    "Validation Mean AUC": MODEL_RESULTS[
                        model_name
                    ]["Validation Mean AUC"],
                    "Test Mean AUC": MODEL_RESULTS[
                        model_name
                    ]["Test Mean AUC"],
                    "Test Mean Accuracy": MODEL_RESULTS[
                        model_name
                    ]["Test Mean Accuracy"],
                })

        hybrid_ablation_table = pd.DataFrame(
            hybrid_ablation_rows
        )

        hybrid_ablation_table.to_csv(
            OUTPUT_DIR / "hybrid_ablation_auc_accuracy.csv",
            index=False,
        )

        display(
            hybrid_ablation_table.style
            .format({
                "Validation Mean AUC": "{:.4f}",
                "Test Mean AUC": "{:.4f}",
                "Test Mean Accuracy": "{:.4f}",
            })
            .set_caption(
                "Hybrid Ablation — ROC-AUC and Accuracy"
            )
        )
else:
    print(
        "Boosted DenseNet121 hybrid was not executed."
    )

## Model 8 — Regularized Multi-channel 2D CNN

Each molecule is represented as six 48 x 48 physicochemical grids: hydrogen bonding, hydrophobicity, metallicity, excluded volume, positive ionization and negative ionization. The implementation remains an RDKit-based open-source approximation of the paper workflow.

In [ ]:
GRID_SIZE = 48
GRID_HALF = 12.0
GRID_AXIS = np.linspace(
    -GRID_HALF,
    GRID_HALF,
    GRID_SIZE,
    dtype=np.float32,
)

GRID_X, GRID_Y = np.meshgrid(
    GRID_AXIS,
    GRID_AXIS,
    indexing="xy",
)

PERIODIC_TABLE = Chem.GetPeriodicTable()
FEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(
    str(Path(RDConfig.RDDataDir) / "BaseFeatures.fdef")
)

METALS = {
    3, 4, 11, 12, 13, 19, 20, 21, 22, 23, 24, 25, 26, 27,
    28, 29, 30, 31, 37, 38, 47, 48, 55, 56, 78, 79, 80,
}

HYDROPHOBIC_ATOMS = {6, 9, 16, 17, 35, 53}

In [ ]:
def molecule_six_channel_grid(mol):
    mol = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol)

    conformer = mol.GetConformer()
    coordinates = np.asarray([
        [
            conformer.GetAtomPosition(i).x,
            conformer.GetAtomPosition(i).y,
        ]
        for i in range(mol.GetNumAtoms())
    ], dtype=np.float32)

    coordinates -= coordinates.mean(axis=0, keepdims=True)

    span = max(
        np.ptp(coordinates[:, 0]),
        np.ptp(coordinates[:, 1]),
        1.0,
    )

    coordinates *= min(1.0, 20.0 / span)

    donor_ids = set()
    acceptor_ids = set()

    for feature in FEATURE_FACTORY.GetFeaturesForMol(mol):
        if feature.GetFamily() == "Donor":
            donor_ids.update(feature.GetAtomIds())
        if feature.GetFamily() == "Acceptor":
            acceptor_ids.update(feature.GetAtomIds())

    channels = np.zeros(
        (6, GRID_SIZE, GRID_SIZE),
        dtype=np.float32,
    )

    for atom_index, atom in enumerate(mol.GetAtoms()):
        x, y = coordinates[atom_index]

        distance = np.sqrt(
            (GRID_X - x) ** 2
            + (GRID_Y - y) ** 2
        ) + 0.25

        atomic_number = atom.GetAtomicNum()
        vdw_radius = float(
            PERIODIC_TABLE.GetRvdw(atomic_number) or 1.7
        )

        vdw_field = 1.0 - np.exp(
            -np.clip((vdw_radius / distance) ** 12, 0, 50)
        )

        if atom_index in donor_ids or atom_index in acceptor_ids:
            c_value, d_value = (
                (1.0, 1.2)
                if atomic_number in (7, 8)
                else (0.8, 1.0)
            )

            h_bond_field = np.abs(
                c_value / np.clip(distance, 0.65, None) ** 12
                - d_value / np.clip(distance, 0.65, None) ** 10
            )

            channels[0] += np.clip(h_bond_field, 0, 10)

        if atomic_number in HYDROPHOBIC_ATOMS:
            channels[1] += vdw_field

        if atomic_number in METALS:
            channels[2] += vdw_field

        channels[3] += vdw_field

        if (
            atom.GetFormalCharge() > 0
            or (
                atomic_number in (7, 15)
                and atom.GetTotalDegree() >= 4
            )
        ):
            channels[4] += vdw_field

        if (
            atom.GetFormalCharge() < 0
            or (
                atomic_number in (8, 16)
                and atom.GetTotalDegree() == 1
            )
        ):
            channels[5] += vdw_field

    for channel_index in range(6):
        maximum = float(
            np.percentile(channels[channel_index], 99.5)
        )

        if maximum > 0:
            channels[channel_index] = np.clip(
                channels[channel_index] / maximum,
                0,
                1,
            )

    return channels.astype(np.float16)

In [ ]:
grid_path = FEATURE_DIR / "six_channel_grids_float16.npy"
grid_progress_path = RESUME_DIR / "six_channel_grid_progress.json"
expected_grid_shape = (len(df), 6, GRID_SIZE, GRID_SIZE)

grid_progress = _ultra_read_json(grid_progress_path, {})

if grid_path.exists():
    try:
        X_grids = np.lib.format.open_memmap(
            grid_path,
            mode="r+",
            dtype="float16",
            shape=expected_grid_shape,
        )
    except Exception:
        grid_path.unlink(missing_ok=True)
        X_grids = np.lib.format.open_memmap(
            grid_path,
            mode="w+",
            dtype="float16",
            shape=expected_grid_shape,
        )
else:
    X_grids = np.lib.format.open_memmap(
        grid_path,
        mode="w+",
        dtype="float16",
        shape=expected_grid_shape,
    )

completed_rows = int(grid_progress.get("completed_rows", 0))
completed_rows = max(0, min(completed_rows, len(df)))

for index in tqdm(
    range(completed_rows, len(df)),
    desc="Generating six-channel molecular grids",
    initial=completed_rows,
    total=len(df),
):
    X_grids[index] = molecule_six_channel_grid(df.iloc[index]["mol"])

    if (index + 1) % 32 == 0 or index + 1 == len(df):
        X_grids.flush()
        _ultra_atomic_write_json(
            grid_progress_path,
            {
                "completed_rows": index + 1,
                "total_rows": len(df),
                "completed": index + 1 == len(df),
            },
        )

X_grids.flush()
X_grids = np.load(grid_path, mmap_mode="r")

print("Grid tensor shape:", X_grids.shape)

In [ ]:
channel_names = [
    "Hydrogen Bond",
    "Hydrophobicity",
    "Metallicity",
    "Excluded Volume",
    "Positive Ionization",
    "Negative Ionization",
]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
sample_grid = np.asarray(X_grids[0], dtype=np.float32)

for ax, grid, name in zip(axes.flat, sample_grid, channel_names):
    image = ax.imshow(grid, cmap="magma", origin="lower")
    ax.set_title(name, fontweight="bold")
    ax.axis("off")
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    "Multi-channel 2D Molecular Grid",
    fontweight="bold",
    fontsize=15,
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "03_multichannel_grid_sample.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

In [ ]:
def translate_grid_with_zeros(grid, shift_y, shift_x):
    translated = np.zeros_like(grid)

    source_y_start = max(0, -shift_y)
    source_y_end = min(grid.shape[1], grid.shape[1] - shift_y)
    source_x_start = max(0, -shift_x)
    source_x_end = min(grid.shape[2], grid.shape[2] - shift_x)

    target_y_start = max(0, shift_y)
    target_y_end = target_y_start + (source_y_end - source_y_start)
    target_x_start = max(0, shift_x)
    target_x_end = target_x_start + (source_x_end - source_x_start)

    translated[
        :,
        target_y_start:target_y_end,
        target_x_start:target_x_end,
    ] = grid[
        :,
        source_y_start:source_y_end,
        source_x_start:source_x_end,
    ]

    return translated

In [ ]:
class GridDataset(Dataset):
    def __init__(self, grid_array, indices, labels, augment=False):
        self.grid_array = grid_array
        self.indices = np.asarray(indices, dtype=int)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        grid = np.asarray(
            self.grid_array[self.indices[index]],
            dtype=np.float32,
        ).copy()

        if self.augment:
            rotation = np.random.randint(0, 4)
            grid = np.rot90(
                grid,
                k=rotation,
                axes=(1, 2),
            ).copy()

            if np.random.rand() < 0.5:
                grid = grid[:, :, ::-1].copy()

            shift_y = np.random.randint(-2, 3)
            shift_x = np.random.randint(-2, 3)
            grid = translate_grid_with_zeros(
                grid,
                shift_y,
                shift_x,
            )

            channel_scale = np.random.uniform(
                0.92,
                1.08,
                size=(6, 1, 1),
            ).astype(np.float32)

            grid *= channel_scale

            if np.random.rand() < 0.40:
                grid += np.random.normal(
                    0.0,
                    0.008,
                    size=grid.shape,
                ).astype(np.float32)

            grid = np.clip(grid, 0.0, 1.0)

        return (
            torch.from_numpy(grid),
            torch.from_numpy(self.labels[index]),
        )

In [ ]:
def norm2d(channels):
    groups = 8 if channels % 8 == 0 else 4
    return nn.GroupNorm(groups, channels)


class MultiChannel2DCNN(nn.Module):
    def __init__(self, number_of_tasks):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(6, 24, 3, padding=1, bias=False),
            norm2d(24),
            nn.SiLU(),
            nn.Dropout2d(0.08),
            nn.MaxPool2d(2),

            nn.Conv2d(24, 48, 3, padding=1, bias=False),
            norm2d(48),
            nn.SiLU(),
            nn.Dropout2d(0.10),
            nn.MaxPool2d(2),

            nn.Conv2d(48, 96, 3, padding=1, bias=False),
            norm2d(96),
            nn.SiLU(),
            nn.Dropout2d(0.12),
            nn.MaxPool2d(2),

            nn.Conv2d(96, 128, 3, padding=1, bias=False),
            norm2d(128),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.45),
            nn.Linear(128, 96),
            nn.SiLU(),
            nn.Dropout(0.35),
            nn.Linear(96, number_of_tasks),
        )

    def forward(self, X):
        return self.head(self.features(X))

In [ ]:
@torch.no_grad()
def predict_grid_tta(
    model,
    indices,
    labels,
    rotations=(0, 1, 2, 3),
):
    dataset = GridDataset(
        X_grids,
        indices,
        labels,
        augment=False,
    )

    loader = DataLoader(
        dataset,
        batch_size=max(BATCH_IMAGE, 24),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    model.eval()
    predictions = []

    for X_batch, _ in loader:
        X_batch = X_batch.to(DEVICE)
        rotation_predictions = []

        for rotation in rotations:
            rotated = torch.rot90(
                X_batch,
                k=rotation,
                dims=(2, 3),
            )

            rotation_predictions.append(
                torch.sigmoid(model(rotated))
            )

        predictions.append(
            torch.stack(
                rotation_predictions,
                dim=0,
            ).mean(dim=0).cpu().numpy()
        )

    return np.vstack(predictions)

In [ ]:
def build_cnn_sampler():
    endpoint_positive = np.nansum(Y_train == 1, axis=0)
    endpoint_negative = np.nansum(Y_train == 0, axis=0)
    rarity = endpoint_negative / np.maximum(endpoint_positive, 1)

    sample_weights = np.ones(len(train_idx), dtype=np.float64)

    for i in range(len(train_idx)):
        positive_tasks = np.where(Y_train[i] == 1)[0]

        if len(positive_tasks):
            moderate_weight = np.sqrt(
                np.minimum(rarity[positive_tasks], 16.0)
            )

            sample_weights[i] += min(
                float(np.mean(moderate_weight)),
                4.0,
            )

    return WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

In [ ]:
def train_grid_cnn():
    model_name = "Regularized Multi-channel 2D CNN"

    restored = restore_completed_model_result(model_name)
    if restored is not None:
        return restored

    train_dataset = GridDataset(
        X_grids,
        train_idx,
        Y_train,
        augment=True,
    )

    validation_dataset = GridDataset(
        X_grids,
        val_idx,
        Y_val,
        augment=False,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_IMAGE,
        sampler=build_cnn_sampler(),
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=max(BATCH_IMAGE, 24),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    model = MultiChannel2DCNN(len(TARGETS)).to(DEVICE)

    pos_weight = compute_pos_weight(
        Y_train,
        cap=6.0,
        power=0.50,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=5e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": [],
        "learning_rate": [],
    }

    best_auc = -np.inf
    wait = 0
    top_states = []
    max_epochs = 120
    patience = 20
    top_k = 3
    start_epoch = 0

    checkpoint_dir = RESUME_DIR / slugify(model_name)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "training_checkpoint.pt"

    if checkpoint_path.exists():
        try:
            checkpoint = torch.load(
                checkpoint_path,
                map_location=DEVICE,
                weights_only=False,
            )
            model.load_state_dict(checkpoint["model_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            scheduler.load_state_dict(checkpoint["scheduler_state"])
            history = checkpoint["history"]
            best_auc = float(checkpoint["best_auc"])
            wait = int(checkpoint["wait"])
            top_states = checkpoint["top_states"]
            start_epoch = int(checkpoint["next_epoch"])
            print(f"Resuming {model_name} from epoch {start_epoch + 1}.")
        except Exception as exc:
            print(
                f"CNN checkpoint could not be restored; training will "
                f"restart. Details: {exc}"
            )
            start_epoch = 0

    for epoch in range(start_epoch, max_epochs):
        model.train()
        train_losses = []

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            Y_batch = Y_batch.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(X_batch)

            loss = masked_focal_bce(
                logits,
                Y_batch,
                pos_weight=pos_weight,
                gamma=1.25,
                label_smoothing=0.01,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )
            optimizer.step()

            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        validation_losses = []
        validation_probabilities = []
        validation_targets = []

        with torch.no_grad():
            for X_batch, Y_batch in validation_loader:
                X_batch = X_batch.to(DEVICE)
                Y_batch = Y_batch.to(DEVICE)

                logits = model(X_batch)

                validation_losses.append(float(
                    masked_focal_bce(
                        logits,
                        Y_batch,
                        pos_weight=pos_weight,
                        gamma=1.25,
                        label_smoothing=0.01,
                    ).cpu()
                ))

                validation_probabilities.append(
                    torch.sigmoid(logits).cpu().numpy()
                )

                validation_targets.append(
                    Y_batch.cpu().numpy()
                )

        train_loss = float(np.mean(train_losses))
        validation_loss = float(np.mean(validation_losses))
        validation_probability = np.vstack(validation_probabilities)
        validation_target = np.vstack(validation_targets)

        validation_auc = mean_metric([
            safe_auc(
                validation_target[:, j],
                validation_probability[:, j],
            )
            for j in range(len(TARGETS))
        ])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(validation_loss)
        history["val_auc"].append(validation_auc)
        history["learning_rate"].append(
            float(optimizer.param_groups[0]["lr"])
        )

        scheduler.step(validation_auc)

        top_states.append({
            "epoch": epoch + 1,
            "val_auc": float(validation_auc),
            "state_dict": clone_state_dict(model),
        })

        top_states = sorted(
            top_states,
            key=lambda item: item["val_auc"],
            reverse=True,
        )[:top_k]

        if validation_auc > best_auc + 2e-4:
            best_auc = validation_auc
            wait = 0
        else:
            wait += 1

        ultra_atomic_torch_save(
            {
                "model_state": clone_state_dict(model),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "history": history,
                "best_auc": best_auc,
                "wait": wait,
                "top_states": top_states,
                "next_epoch": epoch + 1,
            },
            checkpoint_path,
        )

        print(
            f"Epoch {epoch + 1:03d}/{max_epochs} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={validation_loss:.5f} | "
            f"val_auc={validation_auc:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if epoch + 1 >= 20 and wait >= patience:
            print("Validation-AUC early stopping activated.")
            break

    averaged_state = average_state_dicts([
        item["state_dict"]
        for item in top_states
    ])

    model.load_state_dict(averaged_state)
    model = model.to(DEVICE)

    print(
        "CNN top checkpoint epochs:",
        [
            (item["epoch"], round(item["val_auc"], 4))
            for item in top_states
        ],
    )

    P_train = predict_grid_tta(
        model,
        train_idx,
        Y_train,
        rotations=(0, 1),
    )

    P_validation = predict_grid_tta(
        model,
        val_idx,
        Y_val,
        rotations=(0, 1, 2, 3),
    )

    P_test = predict_grid_tta(
        model,
        test_idx,
        Y_test,
        rotations=(0, 1, 2, 3),
    )

    history["best_val_auc"] = float(best_auc)
    history["top_checkpoint_records"] = [
        {
            "epoch": item["epoch"],
            "val_auc": item["val_auc"],
        }
        for item in top_states
    ]

    torch.save(
        model.state_dict(),
        MODEL_DIR / "multichannel_2d_cnn.pt",
    )

    result = register_model_result(
        model_name,
        P_train,
        P_validation,
        P_test,
        history=history,
        diagnostic_method=(
            "Regularized train prediction with two-rotation TTA; "
            "validation-AUC early stopping and checkpoint averaging"
        ),
        notes=(
            "Six-channel 24A molecular-grid CNN with zero-padded translation, "
            "GroupNorm, spatial dropout, moderate sampling, focal BCE, "
            "geometric augmentation, validation-AUC early stopping, "
            "checkpoint averaging and four-rotation validation/test TTA."
        ),
    )

    checkpoint_path.unlink(missing_ok=True)
    return result


In [ ]:
cnn_summary, cnn_per_endpoint = train_grid_cnn()

## Model 9 — Strongly Regularized Compact MTL-DNN with Task-Specific Heads

The previous fully shared `1024-512-128` architecture overfit.

The revised architecture is:

`Filtered ECFP4 + 7 descriptors -> 256 -> 128 shared trunk -> twelve 32-unit task heads -> 12 logits`

This reduces shared capacity while allowing each endpoint to learn a small specialized boundary. Strong dropout, weight decay, strict rare-bit filtering, masked focal BCE, validation-AUC early stopping and checkpoint averaging are retained.

In [ ]:
MTL_DESCRIPTOR_FUNCTIONS = [
    ("MolWt", Descriptors.MolWt),
    ("MolLogP", Crippen.MolLogP),
    ("HBD", Lipinski.NumHDonors),
    ("HBA", Lipinski.NumHAcceptors),
    ("TPSA", rdMolDescriptors.CalcTPSA),
    ("RotB", Lipinski.NumRotatableBonds),
    ("RingCount", rdMolDescriptors.CalcNumRings),
]

X_mtl_descriptors = np.asarray([
    [
        float(function(mol))
        for _, function in MTL_DESCRIPTOR_FUNCTIONS
    ]
    for mol in tqdm(df["mol"], desc="Compact MTL descriptors")
], dtype=np.float32)

In [ ]:
mtl_imputer = SimpleImputer(strategy="median")
mtl_scaler = StandardScaler()

D_train = mtl_scaler.fit_transform(
    mtl_imputer.fit_transform(
        X_mtl_descriptors[train_idx]
    )
)

D_val = mtl_scaler.transform(
    mtl_imputer.transform(
        X_mtl_descriptors[val_idx]
    )
)

D_test = mtl_scaler.transform(
    mtl_imputer.transform(
        X_mtl_descriptors[test_idx]
    )
)

M_train = X_morgan[train_idx].astype(np.float32)
M_val = X_morgan[val_idx].astype(np.float32)
M_test = X_morgan[test_idx].astype(np.float32)

bit_count = np.sum(M_train > 0.5, axis=0)

MTL_KEEP_MASK = (
    (bit_count >= 10)
    & (
        bit_count
        <= int(len(M_train) * 0.995)
    )
)

X_train_mtl = np.hstack([
    M_train[:, MTL_KEEP_MASK],
    D_train,
]).astype(np.float32)

X_val_mtl = np.hstack([
    M_val[:, MTL_KEEP_MASK],
    D_val,
]).astype(np.float32)

X_test_mtl = np.hstack([
    M_test[:, MTL_KEEP_MASK],
    D_test,
]).astype(np.float32)

retained_mtl_bits = int(
    MTL_KEEP_MASK.sum()
)

SPARSE_FEATURE_AUDIT.append({
    "Feature Set": "Compact MTL ECFP4",
    "Input Binary Features": int(N_BITS),
    "Retained Binary Features": retained_mtl_bits,
    "Removed Binary Features": int(
        N_BITS - retained_mtl_bits
    ),
    "Minimum Count": 10,
    "Maximum Fraction": 0.995,
})

print(
    f"Morgan bits retained: "
    f"{retained_mtl_bits} / {N_BITS}"
)

print(
    "Compact MTL input shape:",
    X_train_mtl.shape,
)

In [ ]:
joblib.dump({
    "imputer": mtl_imputer,
    "scaler": mtl_scaler,
    "descriptor_names": [
        item[0]
        for item in MTL_DESCRIPTOR_FUNCTIONS
    ],
    "morgan_keep_mask": MTL_KEEP_MASK,
}, MODEL_DIR / "compact_mtl_preprocessing.joblib")

In [ ]:
class CompactMTLDNN(nn.Module):
    def __init__(
        self,
        input_dimension,
        number_of_tasks,
    ):
        super().__init__()

        self.shared_trunk = nn.Sequential(
            nn.Linear(input_dimension, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.50),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.45),
        )

        self.task_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(128, 32),
                nn.ReLU(),
                nn.Dropout(0.30),
                nn.Linear(32, 1),
            )
            for _ in range(number_of_tasks)
        ])

    def forward(self, X):
        shared = self.shared_trunk(X)

        logits = [
            head(shared)
            for head in self.task_heads
        ]

        return torch.cat(
            logits,
            dim=1,
        )

In [ ]:
mtl_model = CompactMTLDNN(
    X_train_mtl.shape[1],
    len(TARGETS),
).to(DEVICE)

mtl_pos_weight = compute_pos_weight(
    Y_train,
    cap=8.0,
    power=0.65,
)

mtl_optimizer = torch.optim.AdamW(
    mtl_model.parameters(),
    lr=1.8e-4,
    weight_decay=9e-4,
)

mtl_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    mtl_optimizer,
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)

In [ ]:
mtl_summary, mtl_per_endpoint = train_tabular_multitask(
    "Strongly Regularized Compact MTL-DNN",
    mtl_model,
    X_train_mtl,
    Y_train,
    X_val_mtl,
    Y_val,
    X_test_mtl,
    optimizer=mtl_optimizer,
    max_epochs=180,
    batch_size=BATCH_TABULAR,
    patience=20,
    minimum_epochs=15,
    loss_function=lambda raw, y: masked_focal_bce(
        raw,
        y,
        pos_weight=mtl_pos_weight,
        gamma=0.75,
        label_smoothing=0.02,
    ),
    scheduler=mtl_scheduler,
    top_k=3,
    notes=(
        "Compact 256-128 shared trunk with twelve 32-unit task-specific "
        "heads, strict ECFP4 rare-bit filtering, seven descriptors, "
        "LayerNorm, strong dropout, AdamW weight decay, masked focal BCE, "
        "validation-AUC early stopping and checkpoint averaging."
    ),
)

## Model 10 — Validation-Selected Correlation-Aware Weighted Rank Ensemble

This is not hard voting. It combines continuous ranked prediction scores.

For every endpoint:

1. Convert each base model score to a 0-1 rank.
2. Measure validation ROC-AUC.
3. Remove weak models.
4. Remove near-duplicate candidates whose validation rank correlation is at least `0.98` with a stronger retained candidate.
5. Greedily add a candidate only when validation ROC-AUC improves.
6. Use repeated selection counts as weights.
7. Apply the fixed `0.50` threshold only when Accuracy is calculated.

The previous same-validation logistic calibration remains removed. Only ROC-AUC and Accuracy are reported.

In [ ]:
def rank01(values):
    values = np.asarray(values, dtype=float)
    output = np.full(len(values), 0.5, dtype=float)
    mask = np.isfinite(values)

    if mask.sum() <= 1:
        return output

    ranks = rankdata(values[mask], method="average")
    output[mask] = (
        (ranks - 1.0)
        / max(len(ranks) - 1.0, 1.0)
    )

    return output

In [ ]:
base_model_names = [
    name
    for name in MODEL_PREDICTIONS
    if "Rank Ensemble" not in name
]

if len(base_model_names) < 2:
    raise RuntimeError(
        "At least two completed base models are required "
        "for the ensemble."
    )

print("Base models available for the ensemble:")

for name in base_model_names:
    print("-", name)

In [ ]:
number_of_models = len(base_model_names)
number_of_tasks = len(TARGETS)

ensemble_weights = np.zeros(
    (number_of_models, number_of_tasks),
    dtype=float,
)

P_train_ensemble = np.zeros(
    (len(train_idx), number_of_tasks),
    dtype=float,
)

P_validation_ensemble = np.zeros(
    (len(val_idx), number_of_tasks),
    dtype=float,
)

P_test_ensemble = np.zeros(
    (len(test_idx), number_of_tasks),
    dtype=float,
)

selection_rows = []

In [ ]:
MAX_ENSEMBLE_CORRELATION = 0.98

for j, target in enumerate(TARGETS):
    validation_ranks = np.vstack([
        rank01(
            MODEL_PREDICTIONS[name][
                "validation"
            ][:, j]
        )
        for name in base_model_names
    ])

    train_ranks = np.vstack([
        rank01(
            MODEL_PREDICTIONS[name][
                "diagnostic_train"
            ][:, j]
        )
        for name in base_model_names
    ])

    test_ranks = np.vstack([
        rank01(
            MODEL_PREDICTIONS[name][
                "test"
            ][:, j]
        )
        for name in base_model_names
    ])

    validation_aucs = np.asarray([
        safe_auc(
            Y_val[:, j],
            validation_ranks[model_index],
        )
        for model_index in range(
            number_of_models
        )
    ], dtype=float)

    finite_models = np.where(
        np.isfinite(validation_aucs)
    )[0]

    correlation_filtered_models = []

    if len(finite_models) == 0:
        selected_models = [0]
        correlation_filtered_models = [0]

    else:
        best_model_index = finite_models[
            np.argmax(
                validation_aucs[finite_models]
            )
        ]

        best_auc = validation_aucs[
            best_model_index
        ]

        eligible_models = [
            model_index
            for model_index in finite_models
            if validation_aucs[model_index]
            >= max(0.65, best_auc - 0.05)
        ]

        eligible_models = sorted(
            eligible_models,
            key=lambda index: (
                validation_aucs[index]
            ),
            reverse=True,
        )

        # Keep strong but genuinely different candidates.
        for candidate in eligible_models:
            keep_candidate = True

            for retained in correlation_filtered_models:
                correlation = np.corrcoef(
                    validation_ranks[candidate],
                    validation_ranks[retained],
                )[0, 1]

                if (
                    np.isfinite(correlation)
                    and abs(correlation)
                    >= MAX_ENSEMBLE_CORRELATION
                ):
                    keep_candidate = False
                    break

            if keep_candidate:
                correlation_filtered_models.append(
                    candidate
                )

            if (
                len(correlation_filtered_models)
                >= 5
            ):
                break

        if (
            best_model_index
            not in correlation_filtered_models
        ):
            correlation_filtered_models.insert(
                0,
                best_model_index,
            )

        eligible_models = (
            correlation_filtered_models[:5]
        )

        selected_models = [
            best_model_index
        ]

        current_score = (
            validation_ranks[
                best_model_index
            ].copy()
        )

        current_auc = safe_auc(
            Y_val[:, j],
            current_score,
        )

        for _ in range(1, 6):
            best_candidate = None
            best_candidate_auc = current_auc
            best_candidate_score = None

            for model_index in eligible_models:
                trial_score = (
                    current_score
                    * len(selected_models)
                    + validation_ranks[
                        model_index
                    ]
                ) / (
                    len(selected_models) + 1
                )

                trial_auc = safe_auc(
                    Y_val[:, j],
                    trial_score,
                )

                if (
                    np.isfinite(trial_auc)
                    and trial_auc
                    > best_candidate_auc + 0.001
                ):
                    best_candidate = model_index
                    best_candidate_auc = trial_auc
                    best_candidate_score = (
                        trial_score
                    )

            if best_candidate is None:
                break

            selected_models.append(
                best_candidate
            )

            current_score = (
                best_candidate_score
            )

            current_auc = (
                best_candidate_auc
            )

    selection_counts = Counter(
        selected_models
    )

    for model_index, count in (
        selection_counts.items()
    ):
        ensemble_weights[
            model_index,
            j,
        ] = count / len(selected_models)

    P_train_ensemble[:, j] = np.average(
        train_ranks,
        axis=0,
        weights=ensemble_weights[:, j],
    )

    P_validation_ensemble[:, j] = (
        np.average(
            validation_ranks,
            axis=0,
            weights=ensemble_weights[:, j],
        )
    )

    P_test_ensemble[:, j] = np.average(
        test_ranks,
        axis=0,
        weights=ensemble_weights[:, j],
    )

    selection_rows.append({
        "Endpoint": target,
        "Selected Models": " | ".join(
            (
                f"{base_model_names[index]} "
                f"x{selection_counts[index]}"
            )
            for index in selection_counts
        ),
        "Diverse Candidate Models": " | ".join(
            base_model_names[index]
            for index in correlation_filtered_models
        ),
        "Maximum Allowed Rank Correlation": (
            MAX_ENSEMBLE_CORRELATION
        ),
        "Validation AUC Before Blend": (
            float(np.nanmax(validation_aucs))
            if np.isfinite(
                validation_aucs
            ).any()
            else np.nan
        ),
        "Validation AUC After Blend": safe_auc(
            Y_val[:, j],
            P_validation_ensemble[:, j],
        ),
    })

In [ ]:
ensemble_weight_table = pd.DataFrame(
    ensemble_weights,
    index=base_model_names,
    columns=TARGETS,
)

ensemble_selection_table = pd.DataFrame(selection_rows)

ensemble_weight_table.to_csv(
    OUTPUT_DIR / "validation_selected_rank_weights.csv"
)
ensemble_selection_table.to_csv(
    OUTPUT_DIR / "endpoint_selection_report.csv",
    index=False,
)


display(
    ensemble_weight_table.style
    .format("{:.3f}")
    .background_gradient(cmap="Blues")
    .set_caption(
        "Endpoint-wise Validation-Selected Rank Weights"
    )
)


display(
    ensemble_selection_table.style
    .format({
        "Validation AUC Before Blend": "{:.4f}",
        "Validation AUC After Blend": "{:.4f}",
    })
    .set_caption("Greedy Rank-Ensemble Selection Audit")
)

In [ ]:
ensemble_summary, ensemble_per_endpoint = register_model_result(
    "Validation-Selected Correlation-Aware Rank Ensemble",
    P_train_ensemble,
    P_validation_ensemble,
    P_test_ensemble,
    diagnostic_method=(
        "Weighted ranks of base diagnostic-train predictions; "
        "validation-only model selection with correlation-aware "
        "candidate filtering"
    ),
    notes=(
        "Per-endpoint weak-model filtering, correlation-aware diversity "
        "filtering and greedy weighted rank averaging. No test-label "
        "leakage and no same-validation logistic calibration."
    ),
)

# Final Results and Visualizations

All final tables and figures in this notebook use only ROC-AUC and Accuracy as evaluation measures.

Bootstrap confidence intervals are uncertainty summaries for the same two measures; they are not additional performance metrics.

In [ ]:
summary_df = pd.DataFrame(
    MODEL_RESULTS.values()
).sort_values(
    "Test Mean AUC",
    ascending=False,
).reset_index(drop=True)

endpoint_results_df = pd.concat([
    table.assign(Model=model_name)
    for model_name, table in (
        MODEL_ENDPOINT_RESULTS.items()
    )
], ignore_index=True)

summary_df.to_csv(
    OUTPUT_DIR / "final_model_leaderboard.csv",
    index=False,
)

endpoint_results_df.to_csv(
    OUTPUT_DIR / "final_endpoint_results.csv",
    index=False,
)

leaderboard_columns = [
    "Model",
    "Validation Mean AUC",
    "Test Mean AUC",
    "Test Mean Accuracy",
]

display(
    summary_df[leaderboard_columns]
    .style
    .format({
        "Validation Mean AUC": "{:.4f}",
        "Test Mean AUC": "{:.4f}",
        "Test Mean Accuracy": "{:.4f}",
    })
    .background_gradient(
        subset=["Test Mean AUC"],
        cmap="RdYlGn",
        vmin=0.5,
        vmax=1.0,
    )
    .set_caption(
        "Final Model Leaderboard — ROC-AUC and Accuracy"
    )
)

In [ ]:
if len(summary_df):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(18, 7),
    )

    auc_table = summary_df.sort_values(
        "Test Mean AUC"
    )

    axes[0].barh(
        auc_table["Model"],
        auc_table["Test Mean AUC"],
    )

    axes[0].axvline(
        TARGET_MEAN_AUC,
        linestyle="--",
        label=f"Target = {TARGET_MEAN_AUC:.2f}",
    )

    axes[0].set_xlim(0.5, 1.02)

    axes[0].set_title(
        "Mean Test ROC-AUC",
        fontweight="bold",
    )

    axes[0].legend()

    accuracy_table = summary_df.sort_values(
        "Test Mean Accuracy"
    )

    axes[1].barh(
        accuracy_table["Model"],
        accuracy_table["Test Mean Accuracy"],
    )

    axes[1].set_xlim(0.0, 1.02)

    axes[1].set_title(
        "Mean Test Accuracy",
        fontweight="bold",
    )

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "10_model_comparison_auc_accuracy.png",
        dpi=240,
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
if len(summary_df):
    plt.figure(figsize=(12, 7))

    statuses = summary_df["Fitting Status"].unique().tolist()
    palette = dict(zip(
        statuses,
        sns.color_palette("Set2", n_colors=max(3, len(statuses))),
    ))

    for status, group in summary_df.groupby("Fitting Status"):
        plt.scatter(
            group["Diagnostic Train Mean AUC"],
            group["Validation Mean AUC"],
            s=120,
            label=status,
            color=palette[status],
            edgecolor="black",
            alpha=0.9,
        )

        for _, row in group.iterrows():
            plt.annotate(
                row["Model"],
                (
                    row["Diagnostic Train Mean AUC"],
                    row["Validation Mean AUC"],
                ),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )

    lower_limit = min(
        0.5,
        summary_df[[
            "Diagnostic Train Mean AUC",
            "Validation Mean AUC",
        ]].min().min() - 0.02,
    )

    plt.plot(
        [lower_limit, 1],
        [lower_limit, 1],
        "--",
        label="Diagnostic Train = Validation",
    )

    plt.xlim(lower_limit, 1.01)
    plt.ylim(lower_limit, 1.01)
    plt.xlabel("Diagnostic Train Mean AUC")
    plt.ylabel("Validation Mean AUC")
    plt.title(
        "Fitting Diagnostic after Regularization",
        fontweight="bold",
    )
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "11_fitting_diagnostic.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
if len(endpoint_results_df):
    auc_pivot = endpoint_results_df.pivot(
        index="Endpoint",
        columns="Model",
        values="Test AUC",
    )

    plt.figure(
        figsize=(max(14, 1.25 * len(auc_pivot.columns)), 7)
    )

    sns.heatmap(
        auc_pivot,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=0.5,
        vmax=1.0,
        linewidths=0.4,
        cbar_kws={"label": "Test ROC-AUC"},
    )

    plt.title(
        "Test ROC-AUC Heatmap — Models x Tox21 Endpoints",
        fontweight="bold",
    )
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "12_auc_heatmap.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
if len(endpoint_results_df):
    accuracy_pivot = endpoint_results_df.pivot(
        index="Endpoint",
        columns="Model",
        values="Test Accuracy",
    )

    plt.figure(
        figsize=(
            max(
                14,
                1.25 * len(
                    accuracy_pivot.columns
                ),
            ),
            7,
        )
    )

    sns.heatmap(
        accuracy_pivot,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        vmin=0.0,
        vmax=1.0,
        linewidths=0.4,
        cbar_kws={
            "label": "Test Accuracy"
        },
    )

    plt.title(
        "Test Accuracy Heatmap — Models x Tox21 Endpoints",
        fontweight="bold",
    )

    plt.xticks(
        rotation=35,
        ha="right",
    )

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "13_accuracy_heatmap.png",
        dpi=240,
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
if len(summary_df):
    best_model = summary_df.iloc[0]["Model"]

    best_rows = (
        MODEL_ENDPOINT_RESULTS[
            best_model
        ].copy()
    )

    x_positions = np.arange(
        len(best_rows)
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(18, 6),
    )

    axes[0].bar(
        x_positions,
        best_rows["Test AUC"],
    )

    axes[0].axhline(
        best_rows["Test AUC"].mean(),
        linestyle="--",
        label=(
            f"Mean = "
            f"{best_rows['Test AUC'].mean():.3f}"
        ),
    )

    axes[0].set_ylim(
        0.5,
        1.03,
    )

    axes[0].set_title(
        f"ROC-AUC by Endpoint — {best_model}",
        fontweight="bold",
    )

    axes[1].bar(
        x_positions,
        best_rows["Test Accuracy"],
    )

    axes[1].axhline(
        best_rows["Test Accuracy"].mean(),
        linestyle="--",
        label=(
            f"Mean = "
            f"{best_rows['Test Accuracy'].mean():.3f}"
        ),
    )

    axes[1].set_ylim(
        0.0,
        1.03,
    )

    axes[1].set_title(
        f"Accuracy by Endpoint — {best_model}",
        fontweight="bold",
    )

    for ax in axes:
        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(
            best_rows["Endpoint"],
            rotation=45,
            ha="right",
        )

        ax.legend()

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "14_best_model_endpoint_auc_accuracy.png",
        dpi=240,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Best model by Mean Test AUC:",
        best_model,
    )

In [ ]:
if len(summary_df):
    plt.figure(
        figsize=(10, 7)
    )

    plt.scatter(
        summary_df["Test Mean AUC"],
        summary_df["Test Mean Accuracy"],
        s=120,
        edgecolor="black",
        alpha=0.85,
    )

    for _, row in summary_df.iterrows():
        plt.annotate(
            row["Model"],
            (
                row["Test Mean AUC"],
                row["Test Mean Accuracy"],
            ),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )

    plt.xlabel(
        "Mean Test ROC-AUC"
    )

    plt.ylabel(
        "Mean Test Accuracy"
    )

    plt.title(
        "ROC-AUC versus Accuracy",
        fontweight="bold",
    )

    plt.xlim(
        max(
            0.45,
            summary_df[
                "Test Mean AUC"
            ].min() - 0.03,
        ),
        1.01,
    )

    plt.ylim(
        max(
            0.0,
            summary_df[
                "Test Mean Accuracy"
            ].min() - 0.03,
        ),
        1.01,
    )

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "15_auc_vs_accuracy.png",
        dpi=240,
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
if MODEL_HISTORIES:
    number_of_histories = len(MODEL_HISTORIES)

    fig, axes = plt.subplots(
        number_of_histories,
        1,
        figsize=(12, max(4, 4.0 * number_of_histories)),
        squeeze=False,
    )

    for ax, (model_name, history) in zip(
        axes.flat,
        MODEL_HISTORIES.items(),
    ):
        ax.plot(
            history["train_loss"],
            label="Train Loss",
            linewidth=2,
        )
        ax.plot(
            history["val_loss"],
            label="Validation Loss",
            linewidth=2,
        )

        ax.set_title(model_name, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(alpha=0.25)

        if history.get("val_auc"):
            second_axis = ax.twinx()
            second_axis.plot(
                history["val_auc"],
                linestyle="--",
                linewidth=1.8,
                alpha=0.75,
                label="Validation AUC",
            )
            second_axis.set_ylabel("Validation AUC")
            second_axis.set_ylim(0.45, 1.02)

            best_epoch = int(np.nanargmax(
                np.asarray(history["val_auc"], dtype=float)
            ))

            ax.axvline(
                best_epoch,
                linestyle=":",
                alpha=0.7,
                label=f"Best AUC epoch = {best_epoch + 1}",
            )

            lines_1, labels_1 = ax.get_legend_handles_labels()
            lines_2, labels_2 = second_axis.get_legend_handles_labels()

            ax.legend(
                lines_1 + lines_2,
                labels_1 + labels_2,
                loc="best",
            )
        else:
            ax.legend(loc="best")

    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "16_neural_training_curves.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("No neural training history is available.")

In [ ]:
if CV_TABLES:
    all_cv_rows = pd.concat(
        CV_TABLES.values(),
        ignore_index=True,
    )

    cv_summary = (
        all_cv_rows
        .groupby("Model")
        .agg(
            CV_Mean_AUC=("CV Mean AUC", "mean"),
            CV_Std_AUC=("CV Mean AUC", "std"),
            CV_Mean_Accuracy=("CV Mean Accuracy", "mean"),
        )
        .reset_index()
        .sort_values("CV_Mean_AUC", ascending=False)
    )

    cv_summary.to_csv(
        OUTPUT_DIR / "three_fold_cv_summary.csv",
        index=False,
    )

    display(
        cv_summary.style
        .format({
            "CV_Mean_AUC": "{:.4f}",
            "CV_Std_AUC": "{:.4f}",
            "CV_Mean_Accuracy": "{:.4f}",
        })
        .set_caption("Three-Fold Cross-Validation Summary")
    )

    plt.figure(figsize=(12, 5))
    x_positions = np.arange(len(cv_summary))

    plt.bar(
        x_positions,
        cv_summary["CV_Mean_AUC"],
        yerr=cv_summary["CV_Std_AUC"].fillna(0),
        capsize=5,
    )

    plt.xticks(
        x_positions,
        cv_summary["Model"],
        rotation=25,
        ha="right",
    )
    plt.ylim(0.5, 1.02)
    plt.ylabel("Mean CV AUC")
    plt.title(
        "Three-Fold Cross-Validation — Mean AUC",
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "17_three_fold_cv.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()

## Final export and reproducibility metadata

This section exports:

- The final ROC-AUC and Accuracy leaderboard.
- Endpoint-level ROC-AUC and Accuracy.
- Model predictions.
- Cross-validation tables.
- Bootstrap confidence intervals for ROC-AUC and Accuracy.
- Feature, descriptor, scaffold and preprocessing audits.
- Ensemble weights and selection reports.
- Exact package versions used by the run.
- Environment and run configuration metadata.

The notebook does not claim that the target AUC was reached unless the newly executed results show it.

In [ ]:
import sklearn
import rdkit
import xgboost
import importlib.metadata as importlib_metadata

package_versions = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "rdkit": rdkit.__version__,
    "xgboost": xgboost.__version__,
    "torch": torch.__version__,
    "torchvision_available": TORCHVISION_AVAILABLE,
    "device": str(DEVICE),
    "gpu": (
        torch.cuda.get_device_name(0)
        if DEVICE.type == "cuda"
        else None
    ),
    "ram_gb": round(RAM_GB, 2),
    "resource_mode": RESOURCE_MODE,
}

run_configuration = {
    "notebook": "Updated_Shohoj_Pro_Ultra.ipynb",
    "dataset_path": str(DATA_PATH),
    "dataset_sha256": RAW_SHA256,
    "cleaned_samples": len(df),
    "split": {
        "train": TRAIN_RATIO,
        "validation": VALID_RATIO,
        "test": TEST_RATIO,
    },
    "seed": SEED,
    "metrics": [
        "Mean ROC-AUC",
        "Mean Accuracy",
    ],
    "classification_threshold": THRESHOLD,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "pipeline_version": PIPELINE_VERSION,
    "fitting_diagnostic": (
        "Classical models use 3-fold OOF train predictions. "
        "Neural model diagnostic train predictions are retained for "
        "model-specific overfitting review but are not shown in the "
        "main cross-family leaderboard."
    ),
    "target_mean_auc": TARGET_MEAN_AUC,
    "missing_label_strategy": (
        "Mask and ignore missing targets. "
        "Never impute target labels as 0 or 1."
    ),
    "restart_safe_resume_enabled": True,
    "restart_safe_run_stamp": ULTRA_RUN_STAMP,
    "hybrid_model": (
        "DenseNet121 visual PCA features + filtered chemical "
        "features -> regularized XGBoost"
    ),
    "hybrid_ablation": RUN_HYBRID_ABLATION,
    "caps_pretrainer": CAPS_PRETRAINER,
    "scaffold_group_cv_enabled": RUN_SCAFFOLD_GROUP_CV,
    "packages": package_versions,
    "completed_models": (
        summary_df["Model"].tolist()
    ),
    "generated_at": datetime.now().isoformat(
        timespec="seconds"
    ),
}

(
    OUTPUT_DIR
    / "run_config_and_environment.json"
).write_text(
    json.dumps(
        run_configuration,
        indent=2,
    ),
    encoding="utf-8",
)

endpoint_summary_raw.to_csv(
    OUTPUT_DIR
    / "endpoint_missing_imbalance_summary.csv",
    index=False,
)

split_distribution.to_csv(
    OUTPUT_DIR
    / "split_endpoint_distribution.csv",
    index=False,
)

if SPARSE_FEATURE_AUDIT:
    pd.DataFrame(
        SPARSE_FEATURE_AUDIT
    ).to_csv(
        OUTPUT_DIR
        / "sparse_feature_retention_audit.csv",
        index=False,
    )

if FOLD_FEATURE_AUDIT:
    (
        pd.DataFrame(
            FOLD_FEATURE_AUDIT
        )
        .drop_duplicates(
            subset=["Fit Index Hash"]
        )
        .to_csv(
            OUTPUT_DIR
            / "fold_descriptor_retention_audit.csv",
            index=False,
        )
    )

# Export exact versions from the environment that produced the run.
requirements_lines = []

for pip_name in sorted(
    set(PACKAGE_MAP.values())
):
    try:
        version = importlib_metadata.version(
            pip_name
        )

        requirements_lines.append(
            f"{pip_name}=={version}"
        )

    except importlib_metadata.PackageNotFoundError:
        requirements_lines.append(
            f"# {pip_name} was not found"
        )

(
    OUTPUT_DIR
    / "requirements_locked.txt"
).write_text(
    "\n".join(
        requirements_lines
    )
    + "\n",
    encoding="utf-8",
)

model_card = f"""
# Updated_Shohoj_Pro_Ultra Model Card

## Intended use
Research comparison of machine-learning, deep-learning, boosted hybrid and
rank-ensemble models for the 12 Tox21 toxicity endpoints.

## Evaluation
Only ROC-AUC and Accuracy are reported. Accuracy uses a fixed threshold of
{THRESHOLD:.2f}.

## Important limitations
- The main split is multilabel-stratified, not scaffold-separated.
- The six-channel grid is an RDKit-based approximation.
- DenseNet121 uses ImageNet-pretrained visual features.
- Results must not be claimed before a complete run produces the exported files.
- Predictions are research outputs and are not clinical or regulatory decisions.

## Reproducibility
Dataset SHA256: {RAW_SHA256}
Pipeline version: {PIPELINE_VERSION}
Seed: {SEED}
"""

(
    OUTPUT_DIR
    / "MODEL_CARD.md"
).write_text(
    model_card.strip() + "\n",
    encoding="utf-8",
)

In [ ]:
print("=" * 92)
print(
    "Updated_Shohoj_Pro_Ultra pipeline "
    "export complete."
)
print(
    "Final leaderboard:",
    OUTPUT_DIR
    / "final_model_leaderboard.csv",
)
print(
    "Endpoint results:",
    OUTPUT_DIR
    / "final_endpoint_results.csv",
)
print(
    "Locked requirements:",
    OUTPUT_DIR
    / "requirements_locked.txt",
)
print(
    "Figures:",
    FIGURE_DIR,
)
print(
    "Models:",
    MODEL_DIR,
)
print("=" * 92)

if len(summary_df):
    best_result = summary_df.iloc[0]

    print(
        "Best model:",
        best_result["Model"],
    )

    print(
        f"Mean Test AUC: "
        f"{best_result['Test Mean AUC']:.4f}"
    )

    print(
        f"Mean Test Accuracy: "
        f"{best_result['Test Mean Accuracy']:.4f}"
    )


    if (
        best_result["Test Mean AUC"]
        >= TARGET_MEAN_AUC
    ):
        print(
            "The target Mean AUC of 0.90 "
            "was achieved in this run."
        )
    else:
        print(
            "The target Mean AUC of 0.90 was not "
            "achieved in this run. The reported "
            "result is the measured result."
        )

ultra_mark_run_complete()
print("Restart-safe run marked as completed.")


## Research implementation notes

- Evaluation is intentionally limited to **ROC-AUC** and **Accuracy**.
- Accuracy uses the fixed threshold `0.50` for every endpoint and every model.
- Missing labels are masked and are never converted to 0 or 1.
- Classical models use 3-fold OOF diagnostic predictions.
- The main leaderboard excludes cross-family train-AUC comparison because neural train diagnostics remain in-sample.
- Descriptor imputation, near-constant filtering, correlation filtering, scaling and PCA are train-only and fold-specific.
- The hybrid branch is a boosted feature-fusion model and includes chemical-only, visual-only and fused ablation comparisons.
- DenseNet PCA uses `whiten=False` and a 95%-variance rule capped at 64 components.
- The CapsNet branch uses 13 distinct descriptors and denoising-autoencoder pretraining.
- DeepTox and Compact MTL-DNN use smaller shared trunks with task-specific heads.
- The ensemble filters highly correlated validation-rank candidates before greedy weighting.
- Bootstrap intervals quantify uncertainty for ROC-AUC and Accuracy only.
- The Bemis-Murcko scaffold audit measures scaffold overlap across the main split. Optional scaffold-group CV is included as a separate robustness experiment.
- The six-channel grid remains an RDKit-based approximation and should not be described as an exact reproduction of the proprietary ChemAxon/AutoDock workflow.
- The DenseNet visual branch uses ImageNet-pretrained features, not a chemistry-specific pretrained image encoder.
- External validation and chemistry-specific pretrained encoders require additional datasets or model files and are therefore not executed automatically.

## Remaining extensions that require additional assets or a separate large experiment

The technical review also proposed several research extensions that cannot be executed automatically inside this notebook without additional data, pretrained weights or a much larger experimental budget:

- External validation on harmonized ToxCast, ChEMBL or PubChem endpoints.
- Chemistry-specific pretrained encoders such as ChemBERTa, MolFormer, GROVER or GraphMVP.
- Exact ChemAxon/AutoDock reproduction of the six-channel grid.
- Full neural OOF cross-fitting for every deep model.
- Large nested Bayesian hyperparameter searches for every endpoint.
- 3D multi-conformer grids and geometric neural networks.
- SHAP or fragment-attribution analysis for every model family.

The notebook now includes the most practical code-level improvements from the report: scaffold auditing, optional scaffold-group CV, stronger preprocessing audits, descriptor filtering, DNN task heads, DAE-pretrained CapsNet, hybrid ablation, correlation-aware ensembling, bootstrap confidence intervals and exact environment locking.

These remaining extensions should be treated as separate research experiments rather than silently added to the main pipeline.

## References

1. Mayr, A., Klambauer, G., Unterthiner, T., & Hochreiter, S. (2016). *DeepTox: Toxicity Prediction using Deep Learning*. Frontiers in Environmental Science, 3, 80.
2. Wang, Y., et al. (2021). *Multitask CapsNet: An Imbalanced Data Deep Learning Method for Predicting Toxicants*. ACS Omega, 6, 26545–26555.
3. Yuan, Q., et al. (2019). *Toxicity Prediction Method Based on Multi-Channel Convolutional Neural Network*. Molecules, 24, 3383.
4. User-supplied MTL-DNN paper for the compact multi-task architecture.
5. `Updated_Shohoj.ipynb` and `Updated_Shohoj.pdf`, reviewed as the implementation and result-audit sources for this revision.